# AMD Project Notebook

This notebook was automatically generated from the uploaded AMD project ZIP.

It contains:
- Project structure
- Python source files
- Readme/configuration files

You can execute or modify the code directly inside Jupyter.


## Project Structure

```text
AMD/
  .env
  admin.py
  agents.zip
  config.py
  main.py
  readme.txt
  req.txt
  agents/
    base_agent.py
    data_extraction_agent.py
    doc_utils.py
    enrichment_agent.py
    financial_profile_agent.py
    identity_verification_agent.py
    orchestrator.py
    registry_verification_agent.py
    screening_agent.py
    __init__.py
    __pycache__/
      base_agent.cpython-312.pyc
      data_extraction_agent.cpython-312.pyc
      doc_utils.cpython-312.pyc
      enrichment_agent.cpython-312.pyc
      financial_profile_agent.cpython-312.pyc
      identity_verification_agent.cpython-312.pyc
      orchestrator.cpython-312.pyc
      registry_verification_agent.cpython-312.pyc
      screening_agent.cpython-312.pyc
      __init__.cpython-312.pyc
  db/
    db_manager.py
    generate_dummy_db.py
    users_db.json
    __init__.py
    __pycache__/
      db_manager.cpython-312.pyc
      __init__.cpython-312.pyc
  logs/
    audit_trail.jsonl
  __pycache__/
    config.cpython-312.pyc
```

## File: `admin.py`

In [ ]:
"""
admin.py
Admin panel (Streamlit) for managing the mock "DigiLocker-style" identity
registry database (db/users_db.json).

Allows an administrator to:
 - Browse / search all registry records
 - View a single record's full details
 - Add a new record
 - Update / edit an existing record
 - Delete a record
 - Toggle PEP / Sanctioned / Risk flags quickly
 - Export the database as JSON

Run:  streamlit run admin.py
(Run on a different port than main.py if running both simultaneously, e.g.:
 streamlit run admin.py --server.port 8502)
"""

import json

import pandas as pd
import streamlit as st

from db import (
    list_users,
    get_user,
    add_user,
    update_user,
    delete_user,
    replace_user,
    load_db,
)

st.set_page_config(
    page_title="KYC Registry Admin",
    page_icon="🗄️",
    layout="wide",
    initial_sidebar_state="expanded",
)

st.markdown("""
<style>
.admin-badge {
    display: inline-block;
    padding: 0.15rem 0.6rem;
    border-radius: 0.35rem;
    font-size: 0.8rem;
    font-weight: 600;
    color: white;
    margin-right: 0.3rem;
}
</style>
""", unsafe_allow_html=True)


# --------------------------------------------------------------------------
# Sidebar navigation
# --------------------------------------------------------------------------
with st.sidebar:
    st.title("🗄️ KYC Registry Admin")
    st.caption("Manage the mock DigiLocker-style identity registry")
    page = st.radio(
        "Navigate",
        ["📋 Browse Records", "➕ Add Record", "✏️ Edit / Delete Record", "📤 Export / Raw DB"],
        index=0,
    )
    st.divider()
    db_preview = load_db()
    st.metric("Total Records", len(db_preview))
    st.metric("PEP Flagged", sum(1 for r in db_preview.values() if r.get("is_pep")))
    st.metric("Sanctioned", sum(1 for r in db_preview.values() if r.get("is_sanctioned")))
    st.metric("Risk Flagged", sum(1 for r in db_preview.values() if r.get("risk_flag")))


# --------------------------------------------------------------------------
# Helper: render record form, returns a dict of values
# --------------------------------------------------------------------------
def record_form(prefix: str, existing: dict = None):
    existing = existing or {}
    address = existing.get("address", {}) or {}
    business = existing.get("business", {}) or {}
    income = existing.get("income", {}) or {}

    st.markdown("##### Identity Documents")
    c1, c2 = st.columns(2)
    with c1:
        aadhaar = st.text_input("Aadhaar", value=existing.get("aadhaar") or "", key=f"{prefix}_aadhaar")
        passport = st.text_input("Passport", value=existing.get("passport") or "", key=f"{prefix}_passport")
    with c2:
        pan = st.text_input("PAN", value=existing.get("pan") or "", key=f"{prefix}_pan")
        dl = st.text_input("Driving Licence", value=existing.get("driving_licence") or "", key=f"{prefix}_dl")

    st.markdown("##### Personal Details")
    c1, c2, c3 = st.columns(3)
    with c1:
        name = st.text_input("Full Name", value=existing.get("name") or "", key=f"{prefix}_name")
        nationality = st.text_input("Nationality", value=existing.get("nationality") or "Indian", key=f"{prefix}_nationality")
    with c2:
        dob = st.text_input("Date of Birth (YYYY-MM-DD)", value=existing.get("dob") or "", key=f"{prefix}_dob")
        gender = st.selectbox(
            "Gender", ["Male", "Female", "Other"],
            index=["Male", "Female", "Other"].index(existing.get("gender")) if existing.get("gender") in ["Male", "Female", "Other"] else 0,
            key=f"{prefix}_gender",
        )
    with c3:
        phone = st.text_input("Phone", value=existing.get("phone") or "", key=f"{prefix}_phone")
        email = st.text_input("Email", value=existing.get("email") or "", key=f"{prefix}_email")

    st.markdown("##### Address")
    c1, c2, c3, c4, c5 = st.columns(5)
    with c1:
        line1 = st.text_input("Address Line 1", value=address.get("line1") or "", key=f"{prefix}_line1")
    with c2:
        city = st.text_input("City", value=address.get("city") or "", key=f"{prefix}_city")
    with c3:
        state = st.text_input("State", value=address.get("state") or "", key=f"{prefix}_state")
    with c4:
        pin = st.text_input("PIN", value=address.get("pin") or "", key=f"{prefix}_pin")
    with c5:
        country = st.text_input("Country", value=address.get("country") or "India", key=f"{prefix}_country")

    st.markdown("##### Business Details (optional)")
    c1, c2, c3, c4, c5 = st.columns(5)
    with c1:
        cin = st.text_input("CIN", value=business.get("cin") or "", key=f"{prefix}_cin")
    with c2:
        gst = st.text_input("GST", value=business.get("gst") or "", key=f"{prefix}_gst")
    with c3:
        tan = st.text_input("TAN", value=business.get("tan") or "", key=f"{prefix}_tan")
    with c4:
        biz_name = st.text_input("Business Name", value=business.get("name") or "", key=f"{prefix}_bizname")
    with c5:
        biz_types = ["", "Pvt Ltd", "LLP", "Proprietorship", "Partnership"]
        existing_type = business.get("type") or ""
        biz_type = st.selectbox(
            "Business Type", biz_types,
            index=biz_types.index(existing_type) if existing_type in biz_types else 0,
            key=f"{prefix}_biztype",
        )

    st.markdown("##### Income / Employment")
    c1, c2, c3 = st.columns(3)
    with c1:
        annual = st.number_input(
            "Annual Declared Income", min_value=0,
            value=int(income.get("annual_declared") or 0), step=10000, key=f"{prefix}_annual",
        )
    with c2:
        employer = st.text_input("Employer", value=income.get("employer") or "", key=f"{prefix}_employer")
    with c3:
        emp_types = ["", "Salaried", "Business", "Self-Employed", "Unemployed", "Retired"]
        existing_emp = income.get("employment_type") or ""
        emp_type = st.selectbox(
            "Employment Type", emp_types,
            index=emp_types.index(existing_emp) if existing_emp in emp_types else 0,
            key=f"{prefix}_emptype",
        )

    st.markdown("##### Compliance Flags")
    c1, c2, c3 = st.columns(3)
    with c1:
        is_pep = st.checkbox("Politically Exposed Person (PEP)", value=bool(existing.get("is_pep", False)), key=f"{prefix}_pep")
    with c2:
        is_sanctioned = st.checkbox("Sanctioned", value=bool(existing.get("is_sanctioned", False)), key=f"{prefix}_sanctioned")
    with c3:
        risk_flag = st.checkbox("General Risk Flag", value=bool(existing.get("risk_flag", False)), key=f"{prefix}_riskflag")

    record = {
        "aadhaar": aadhaar or None,
        "pan": pan or None,
        "passport": passport or None,
        "driving_licence": dl or None,
        "name": name or None,
        "dob": dob or None,
        "gender": gender,
        "address": {
            "line1": line1 or None,
            "city": city or None,
            "state": state or None,
            "pin": pin or None,
            "country": country or None,
        },
        "nationality": nationality or None,
        "phone": phone or None,
        "email": email or None,
        "business": {
            "cin": cin or None,
            "gst": gst or None,
            "tan": tan or None,
            "name": biz_name or None,
            "type": biz_type or None,
        },
        "income": {
            "annual_declared": annual,
            "employer": employer or None,
            "employment_type": emp_type or None,
        },
        "is_pep": is_pep,
        "is_sanctioned": is_sanctioned,
        "risk_flag": risk_flag,
    }
    return record


# --------------------------------------------------------------------------
# PAGE: Browse Records
# --------------------------------------------------------------------------
if page == "📋 Browse Records":
    st.header("Browse Identity Registry Records")

    db = load_db()
    search = st.text_input("🔍 Search by name, ID number, email, or phone")

    rows = []
    for rid, rec in list_users(db):
        rows.append({
            "ID": rid,
            "Name": rec.get("name"),
            "Aadhaar": rec.get("aadhaar"),
            "PAN": rec.get("pan"),
            "Passport": rec.get("passport"),
            "DL": rec.get("driving_licence"),
            "DOB": rec.get("dob"),
            "City": (rec.get("address") or {}).get("city"),
            "Phone": rec.get("phone"),
            "Email": rec.get("email"),
            "PEP": "🟡" if rec.get("is_pep") else "",
            "Sanctioned": "🔴" if rec.get("is_sanctioned") else "",
            "Risk": "⚠️" if rec.get("risk_flag") else "",
        })

    df = pd.DataFrame(rows)

    if search:
        s = search.strip().lower()
        mask = df.apply(lambda row: row.astype(str).str.lower().str.contains(s).any(), axis=1)
        df = df[mask]

    st.dataframe(df, use_container_width=True, height=500)
    st.caption(f"Showing {len(df)} of {len(rows)} records.")


# --------------------------------------------------------------------------
# PAGE: Add Record
# --------------------------------------------------------------------------
elif page == "➕ Add Record":
    st.header("Add New Registry Record")

    with st.form("add_record_form"):
        custom_id = st.text_input("Record ID (leave blank to auto-assign next available integer)")
        record = record_form("add")
        submitted = st.form_submit_button("➕ Add Record", type="primary")

    if submitted:
        if not record.get("name"):
            st.error("Name is required.")
        else:
            try:
                rid = add_user(record, record_id=custom_id.strip() or None)
                st.success(f"Record added with ID '{rid}'.")
                st.json(get_user(rid))
            except ValueError as e:
                st.error(str(e))


# --------------------------------------------------------------------------
# PAGE: Edit / Delete Record
# --------------------------------------------------------------------------
elif page == "✏️ Edit / Delete Record":
    st.header("Edit or Delete a Registry Record")

    db = load_db()
    record_ids = [rid for rid, _ in list_users(db)]

    if not record_ids:
        st.info("No records in the database yet.")
    else:
        selected_id = st.selectbox("Select Record ID", record_ids)
        existing = get_user(selected_id, db)

        if existing:
            with st.form("edit_record_form"):
                updated_record = record_form("edit", existing)
                col1, col2 = st.columns(2)
                with col1:
                    save_submitted = st.form_submit_button("💾 Save Changes", type="primary", use_container_width=True)
                with col2:
                    delete_submitted = st.form_submit_button("🗑️ Delete Record", use_container_width=True)

            if save_submitted:
                replace_user(selected_id, updated_record)
                st.success(f"Record '{selected_id}' updated.")
                st.json(get_user(selected_id))

            if delete_submitted:
                delete_user(selected_id)
                st.success(f"Record '{selected_id}' deleted.")
                st.rerun()

            st.divider()
            st.markdown("##### Quick Flag Toggles")
            c1, c2, c3 = st.columns(3)
            with c1:
                if st.button(("Unflag" if existing.get("is_pep") else "Flag") + " as PEP", use_container_width=True):
                    update_user(selected_id, {"is_pep": not existing.get("is_pep", False)})
                    st.rerun()
            with c2:
                if st.button(("Unflag" if existing.get("is_sanctioned") else "Flag") + " as Sanctioned", use_container_width=True):
                    update_user(selected_id, {"is_sanctioned": not existing.get("is_sanctioned", False)})
                    st.rerun()
            with c3:
                if st.button(("Unflag" if existing.get("risk_flag") else "Flag") + " as Risk", use_container_width=True):
                    update_user(selected_id, {"risk_flag": not existing.get("risk_flag", False)})
                    st.rerun()


# --------------------------------------------------------------------------
# PAGE: Export / Raw DB
# --------------------------------------------------------------------------
elif page == "📤 Export / Raw DB":
    st.header("Export / Raw Database View")

    db = load_db()
    st.download_button(
        "Download users_db.json",
        data=json.dumps(db, indent=2, ensure_ascii=False),
        file_name="users_db.json",
        mime="application/json",
    )

    with st.expander("View raw JSON", expanded=False):
        st.json(db)


## File: `agents/__init__.py`

In [ ]:
"""
agents package
Contains all specialized KYC agents orchestrated by the AMD (Agent Master / Director).
"""

from .data_extraction_agent import DataExtractionAgent
from .enrichment_agent import EnrichmentAgent
from .identity_verification_agent import IdentityVerificationAgent
from .registry_verification_agent import RegistryVerificationAgent
from .screening_agent import ScreeningAgent
from .financial_profile_agent import FinancialProfileAgent
from .orchestrator import KYCOrchestrator

__all__ = [
    "DataExtractionAgent",
    "EnrichmentAgent",
    "IdentityVerificationAgent",
    "RegistryVerificationAgent",
    "ScreeningAgent",
    "FinancialProfileAgent",
    "KYCOrchestrator",
]


## File: `agents/base_agent.py`

In [ ]:
"""
base_agent.py
Common base class for all KYC agents: standardized logging, LLM call wrapper,
and explainability evidence formatting.
"""

import json
import time
import traceback
from datetime import datetime, timezone
from typing import Any, Dict, List, Optional

from config import llm, AUDIT_LOG_FILE


class AgentResult:
    """Standardized result object returned by every agent."""

    def __init__(
        self,
        agent_name: str,
        status: str,                 # "success" | "warning" | "error"
        risk_score: float,           # 0-100, higher = riskier
        findings: Dict[str, Any],
        evidence: List[str],
        explanation: str,
        raw_data: Optional[Dict[str, Any]] = None,
    ):
        self.agent_name = agent_name
        self.status = status
        self.risk_score = max(0.0, min(100.0, risk_score))
        self.findings = findings
        self.evidence = evidence
        self.explanation = explanation
        self.raw_data = raw_data or {}
        self.timestamp = datetime.now(timezone.utc).isoformat()

    def to_dict(self) -> Dict[str, Any]:
        return {
            "agent_name": self.agent_name,
            "status": self.status,
            "risk_score": round(self.risk_score, 2),
            "findings": self.findings,
            "evidence": self.evidence,
            "explanation": self.explanation,
            "raw_data": self.raw_data,
            "timestamp": self.timestamp,
        }


class BaseAgent:
    """Base class providing shared helpers for all specialized agents."""

    name: str = "BaseAgent"

    def __init__(self, llm_client=None):
        self.llm = llm_client or llm

    # ------------------------------------------------------------------
    # LLM helper with basic retry & JSON-extraction support
    # ------------------------------------------------------------------
    def call_llm(self, prompt: str, expect_json: bool = False, retries: int = 2) -> str:
        last_err = None
        for attempt in range(retries + 1):
            try:
                response = self.llm.invoke(prompt)
                content = response.content if hasattr(response, "content") else str(response)
                if expect_json:
                    return self._extract_json_block(content)
                return content
            except Exception as e:
                last_err = e
                time.sleep(1.0 * (attempt + 1))
        raise RuntimeError(f"[{self.name}] LLM call failed after retries: {last_err}")

    @staticmethod
    def _extract_json_block(text: str) -> str:
        """Strip markdown code fences and return the raw JSON string."""
        text = text.strip()
        if "```" in text:
            parts = text.split("```")
            for part in parts:
                part = part.strip()
                if part.startswith("json"):
                    part = part[4:].strip()
                if part.startswith("{") or part.startswith("["):
                    return part
        return text

    def safe_json_loads(self, text: str, default: Any = None) -> Any:
        try:
            return json.loads(self._extract_json_block(text))
        except Exception:
            return default if default is not None else {}

    # ------------------------------------------------------------------
    # Audit logging - append-only JSONL for explainability / compliance
    # ------------------------------------------------------------------
    def log_audit(self, customer_id: str, payload: Dict[str, Any]):
        record = {
            "timestamp": datetime.now(timezone.utc).isoformat(),
            "agent": self.name,
            "customer_id": customer_id,
            "payload": payload,
        }
        try:
            with open(AUDIT_LOG_FILE, "a", encoding="utf-8") as f:
                f.write(json.dumps(record, default=str) + "\n")
        except Exception:
            traceback.print_exc()

    # ------------------------------------------------------------------
    # To be implemented by subclasses
    # ------------------------------------------------------------------
    def run(self, *args, **kwargs) -> AgentResult:
        raise NotImplementedError


## File: `agents/data_extraction_agent.py`

In [ ]:
"""
data_extraction_agent.py
Agent 1: Data & Document Extraction.

Responsibilities:
 - OCR / parse uploaded documents (identity proof, address proof, business docs)
 - Use the LLM to extract structured fields (name, DOB, ID number, address, etc.)
 - Cross-check consistency between documents (e.g., name on ID vs address proof)
 - Flag missing/poor-quality documents

Change log:
 - the portrait is auto-extracted from `identity_proof` by IdentityVerificationAgent.
 - `live_photo` still skips OCR (it is a selfie, not a text document) but
   now uses the shared _PHOTO_ONLY_CATEGORIES constant so future additions
   are made in one place.
 - Added image quality check for `identity_proof` in addition to `live_photo`
   so blur / resolution problems on the ID card are surfaced early, before
   the face-extraction step in Agent 3.
 - `mimetype` field (added by main.py) now considered alongside filename
   extension when deciding whether to run OCR.
"""

import json
from typing import Any, Dict, List, Set

from .base_agent import BaseAgent, AgentResult
from .doc_utils import extract_text, get_image_quality_flags

# Categories that carry image content only — OCR is skipped, quality is checked.
# `identity_proof` is NOT here: it is a real document that must be OCR-ed for
# field extraction AND quality-checked for the downstream face-extraction step.
_PHOTO_ONLY_CATEGORIES: Set[str] = {"live_photo", "selfie", "live_selfie", "liveness"}

# Categories where we also run an image quality check (in addition to OCR).
# Catching blur / low-res early avoids silent failures in face extraction.
_QUALITY_CHECK_CATEGORIES: Set[str] = {"identity_proof", "live_photo", "selfie"}

EXTRACTION_PROMPT = """You are a KYC document data-extraction specialist.
Given the raw OCR/text content of a document of type "{doc_type}", extract the
following structured fields as a JSON object. If a field is not present, use null.

Required fields:
- full_name
- date_of_birth (format YYYY-MM-DD if possible)
- id_number (passport/license/govt id number)
- nationality
- address
- document_expiry (YYYY-MM-DD if present)
- issuing_authority
- additional_notes (any anomalies, e.g. expired stamp, tampering signs noticed in text)

Document type: {doc_type}
Raw extracted text:
---
{raw_text}
---

Respond with ONLY a valid JSON object, no commentary, no markdown fences.
"""


class DataExtractionAgent(BaseAgent):
    name = "DataExtractionAgent"

    def run(self, customer_id: str, documents: Dict[str, Dict[str, Any]]) -> AgentResult:
        """
        Parameters
        ----------
        customer_id : str
        documents   : dict keyed by category (e.g. ``'identity_proof'``) →
                      ``{"filename": str, "bytes": bytes, "doc_subtype": str,
                         "mimetype": str}``
                      The ``mimetype`` key is set by main.py; fall back to
                      filename extension if absent.
        """
        extracted: Dict[str, Any] = {}
        evidence: List[str] = []
        quality_issues: List[str] = []
        missing_docs: List[str] = []

        from config import DOCUMENT_CATEGORIES

        for category, meta in DOCUMENT_CATEGORIES.items():
            doc = documents.get(category)

            if not doc or not doc.get("bytes"):
                if meta["required"]:
                    missing_docs.append(meta["label"])
                continue

            filename: str = doc["filename"]
            file_bytes: bytes = doc["bytes"]
            doc_subtype: str = doc.get("doc_subtype", meta["label"])

            # ── Image quality check (runs for photos AND identity docs) ──────
            if category in _QUALITY_CHECK_CATEGORIES:
                quality_flags = get_image_quality_flags(file_bytes)
                if quality_flags:
                    quality_issues.extend(
                        [f"{meta['label']}: {q}" for q in quality_flags]
                    )
            else:
                quality_flags = []

            # ── Photo-only categories: skip OCR, record and move on ──────────
            if category in _PHOTO_ONLY_CATEGORIES:
                evidence.append(
                    f"{meta['label']} ({filename}) received and quality-checked."
                )
                extracted[category] = {
                    "filename": filename,
                    "doc_subtype": doc_subtype,
                    "quality_flags": quality_flags,
                }
                continue

            # ── Text extraction (OCR / PDF parse) ────────────────────────────
            raw_text = extract_text(filename, file_bytes)

            if not raw_text or raw_text.startswith("["):
                quality_issues.append(
                    f"{meta['label']} ({filename}): "
                    f"{raw_text or 'no text extracted'}"
                )
                extracted[category] = {
                    "filename": filename,
                    "doc_subtype": doc_subtype,
                    "raw_text": raw_text,
                    "fields": {},
                    "quality_flags": quality_flags,
                }
                continue

            # ── LLM structured field extraction ──────────────────────────────
            prompt = EXTRACTION_PROMPT.format(
                doc_type=doc_subtype,
                raw_text=raw_text[:6000],
            )
            try:
                response = self.call_llm(prompt, expect_json=True)
                fields = self.safe_json_loads(response, default={})
            except Exception as e:
                fields = {}
                quality_issues.append(
                    f"{meta['label']} ({filename}): extraction LLM error — {e}"
                )

            extracted[category] = {
                "filename": filename,
                "doc_subtype": doc_subtype,
                "raw_text": raw_text[:2000],
                "fields": fields,
                "quality_flags": quality_flags,
            }
            evidence.append(
                f"Extracted structured fields from {meta['label']} ({filename})."
            )

        # ── Cross-document consistency checks ────────────────────────────────
        consistency_issues = self._cross_check(extracted)

        # ── Risk scoring ──────────────────────────────────────────────────────
        risk_score = 0.0
        risk_score += len(missing_docs) * 25
        risk_score += len(quality_issues) * 8
        risk_score += len(consistency_issues) * 15
        risk_score = min(100.0, risk_score)

        if missing_docs:
            status = "error"
        elif quality_issues or consistency_issues:
            status = "warning"
        else:
            status = "success"

        explanation_parts: List[str] = []
        if missing_docs:
            explanation_parts.append(
                f"Missing required documents: {', '.join(missing_docs)}."
            )
        if quality_issues:
            explanation_parts.append(
                f"{len(quality_issues)} document quality issue(s) detected."
            )
        if consistency_issues:
            explanation_parts.append(
                f"{len(consistency_issues)} cross-document consistency issue(s) found."
            )
        if not explanation_parts:
            explanation_parts.append(
                "All required documents received, extracted, and mutually consistent."
            )

        findings = {
            "extracted_data": extracted,
            "missing_documents": missing_docs,
            "quality_issues": quality_issues,
            "consistency_issues": consistency_issues,
        }

        result = AgentResult(
            agent_name=self.name,
            status=status,
            risk_score=risk_score,
            findings=findings,
            evidence=evidence,
            explanation=" ".join(explanation_parts),
            raw_data={"extracted": extracted},
        )
        self.log_audit(customer_id, result.to_dict())
        return result

    @staticmethod
    def _cross_check(extracted: Dict[str, Any]) -> List[str]:
        """
        Compare key fields (name, address) across documents for mismatches.

        Uses a token-overlap heuristic: if fewer than 50 % of name tokens
        are shared between two documents the names are considered inconsistent.
        """
        issues: List[str] = []
        names: List[tuple] = []

        for cat in ("identity_proof", "address_proof", "business_doc"):
            data = extracted.get(cat, {})
            fields = data.get("fields", {}) if isinstance(data, dict) else {}
            if fields.get("full_name"):
                names.append((cat, fields["full_name"].strip().lower()))

        if len(names) >= 2:
            base_cat, base_name = names[0]
            base_tokens = set(base_name.split())
            for cat, name in names[1:]:
                name_tokens = set(name.split())
                if base_tokens and name_tokens:
                    overlap = len(base_tokens & name_tokens) / max(
                        len(base_tokens), len(name_tokens)
                    )
                    if overlap < 0.5:
                        issues.append(
                            f"Name mismatch between {base_cat} "
                            f"('{base_name}') and {cat} ('{name}')."
                        )

        return issues

## File: `agents/doc_utils.py`

In [ ]:
"""
doc_utils.py
Utility functions for reading uploaded documents (images / PDFs),
extracting text via OCR, and basic face-comparison for photo verification.

Change log:
  - Added extract_face_from_id(): automatically detects and crops the portrait
    photo embedded in an identity document (Passport, Aadhaar, DL, etc.) so
    callers no longer need a separately uploaded "Photo on ID" file.
  - Updated compare_faces() to accept either raw bytes OR a pre-cropped
    numpy BGR array, enabling the identity_verification_agent to pass the
    result of extract_face_from_id() directly without re-encoding.
"""

import io
import logging
import os
from typing import Optional, Tuple, Union

import numpy as np
from PIL import Image

logger = logging.getLogger(__name__)

try:
    import pytesseract
    TESSERACT_AVAILABLE = True
except ImportError:
    TESSERACT_AVAILABLE = False

try:
    import fitz  # PyMuPDF
    PYMUPDF_AVAILABLE = True
except ImportError:
    PYMUPDF_AVAILABLE = False

try:
    import cv2
    CV2_AVAILABLE = True
except ImportError:
    CV2_AVAILABLE = False


# ---------------------------------------------------------------------------
# Internal helpers
# ---------------------------------------------------------------------------

def _bytes_to_bgr(data: Union[bytes, bytearray, np.ndarray, Image.Image]) -> Optional[np.ndarray]:
    """
    Normalise any supported image representation to an OpenCV BGR numpy array.

    Accepts:
      - bytes / bytearray  : raw encoded image bytes (JPEG, PNG, …)
      - numpy.ndarray      : already-decoded BGR array (returned as-is / copy)
      - PIL.Image.Image    : converted via RGB → BGR
    Returns None on failure.
    """
    if not CV2_AVAILABLE:
        return None

    if isinstance(data, (bytes, bytearray)):
        arr = np.frombuffer(data, np.uint8)
        img = cv2.imdecode(arr, cv2.IMREAD_COLOR)
        return img  # may be None if decoding fails

    if isinstance(data, np.ndarray):
        return data.copy()

    if isinstance(data, Image.Image):
        rgb = np.array(data.convert("RGB"))
        return cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)

    logger.warning("_bytes_to_bgr: unsupported type %s", type(data))
    return None


def _load_face_cascade() -> Optional["cv2.CascadeClassifier"]:
    if not CV2_AVAILABLE:
        return None
    cascade_path = cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
    if not os.path.exists(cascade_path):
        logger.warning("Haar cascade XML not found at %s", cascade_path)
        return None
    return cv2.CascadeClassifier(cascade_path)


def _run_face_detection(img_bgr: np.ndarray) -> list:
    """
    Run Haar-cascade face detection on a BGR image.
    Returns a list of (x, y, w, h) tuples (may be empty).
    Applies histogram equalisation on the gray channel before detection
    to improve robustness on ID-document scans.
    """
    cascade = _load_face_cascade()
    if cascade is None:
        return []

    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    gray = cv2.equalizeHist(gray)
    faces = cascade.detectMultiScale(
        gray,
        scaleFactor=1.05,
        minNeighbors=4,
        minSize=(60, 60),
        flags=cv2.CASCADE_SCALE_IMAGE,
    )
    return list(faces) if len(faces) else []


# ---------------------------------------------------------------------------
# Text extraction
# ---------------------------------------------------------------------------

def extract_text_from_image(file_bytes: bytes) -> str:
    """Run OCR on an image and return extracted text."""
    if not TESSERACT_AVAILABLE:
        return "[OCR unavailable: pytesseract / tesseract-ocr not installed]"
    try:
        img = Image.open(io.BytesIO(file_bytes))
        if img.mode != "RGB":
            img = img.convert("RGB")
        text = pytesseract.image_to_string(img)
        return text.strip()
    except Exception as e:
        return f"[OCR error: {e}]"


def extract_text_from_pdf(file_bytes: bytes) -> str:
    """Extract text from a PDF. Falls back to OCR per-page if no text layer."""
    if not PYMUPDF_AVAILABLE:
        return "[PDF parsing unavailable: PyMuPDF not installed]"
    text_chunks = []
    try:
        doc = fitz.open(stream=file_bytes, filetype="pdf")
        for page in doc:
            page_text = page.get_text().strip()
            if page_text:
                text_chunks.append(page_text)
            elif TESSERACT_AVAILABLE:
                pix = page.get_pixmap(matrix=fitz.Matrix(2, 2))
                img_bytes = pix.tobytes("png")
                ocr_text = extract_text_from_image(img_bytes)
                text_chunks.append(ocr_text)
        doc.close()
    except Exception as e:
        return f"[PDF error: {e}]"
    return "\n".join(text_chunks).strip()


def extract_text(filename: str, file_bytes: bytes) -> str:
    """Dispatch text extraction based on file extension."""
    ext = os.path.splitext(filename)[1].lower()
    if ext == ".pdf":
        return extract_text_from_pdf(file_bytes)
    if ext in (".png", ".jpg", ".jpeg", ".bmp", ".tiff", ".webp"):
        return extract_text_from_image(file_bytes)
    if ext in (".txt", ".csv"):
        try:
            return file_bytes.decode("utf-8", errors="ignore")
        except Exception:
            return ""
    return f"[Unsupported file type: {ext}]"


# ---------------------------------------------------------------------------
# Face extraction from identity documents  (NEW)
# ---------------------------------------------------------------------------

def extract_face_from_id(
    image_input: Union[bytes, bytearray, np.ndarray, Image.Image, str]
) -> Optional[np.ndarray]:
    """
    Automatically detect and crop the portrait photo from an identity
    document (Passport, Aadhaar card, Driving Licence, National ID, etc.).

    This replaces the previous workflow that required users to manually upload
    a cropped "Photo on ID Proof" file.

    Detection strategy
    ------------------
    Pass 1 — standard Haar cascade on the equalised grayscale image.
    Pass 2 — if Pass 1 finds nothing, apply CLAHE contrast enhancement and retry.
              CLAHE helps with washed-out or unevenly lit scanned documents.

    The largest detected face region is returned (ID cards carry exactly one
    portrait; picking the largest face discards small background artefacts).
    A 15 % padding is added on all sides so the crop includes forehead / chin
    context that improves downstream histogram-correlation matching.

    Args:
        image_input:
            Any of the following:
              - ``bytes`` / ``bytearray``  — raw encoded image bytes
              - ``numpy.ndarray``          — BGR image array (OpenCV convention)
              - ``PIL.Image.Image``        — PIL image object
              - ``str``                    — file-system path to an image file

            PDF inputs are NOT supported here; rasterise individual pages with
            PyMuPDF first (see ``pdf_page_to_bgr`` below) and pass the result.

    Returns:
        numpy.ndarray (BGR, uint8) of the cropped face region,
        or ``None`` if detection failed (low quality scan, text-only PDF page,
        no face present, or OpenCV not installed).
    """
    if not CV2_AVAILABLE:
        logger.warning("extract_face_from_id: OpenCV not available — cannot extract face")
        return None

    # ── Normalise input ────────────────────────────────────────────────────
    if isinstance(image_input, str):
        # File-system path
        img = cv2.imread(image_input)
        if img is None:
            logger.warning("extract_face_from_id: could not read file at '%s'", image_input)
            return None
    else:
        img = _bytes_to_bgr(image_input)

    if img is None or img.size == 0:
        logger.warning("extract_face_from_id: received empty or unreadable image")
        return None

    # ── Pass 1: standard detection ─────────────────────────────────────────
    faces = _run_face_detection(img)

    # ── Pass 2: CLAHE enhancement + retry ─────────────────────────────────
    if not faces:
        logger.info(
            "extract_face_from_id: no face in Pass 1 — retrying with CLAHE enhancement"
        )
        lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
        l_ch, a_ch, b_ch = cv2.split(lab)
        clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
        l_ch = clahe.apply(l_ch)
        enhanced = cv2.cvtColor(cv2.merge([l_ch, a_ch, b_ch]), cv2.COLOR_LAB2BGR)
        faces = _run_face_detection(enhanced)
        if faces:
            img = enhanced      # crop from the enhanced image for better quality

    if not faces:
        logger.warning(
            "extract_face_from_id: could not detect a face in the identity document "
            "(document may be low-resolution, a text-only PDF page, or the portrait "
            "region is not detectable by the Haar cascade)"
        )
        return None

    # ── Pick the largest face ──────────────────────────────────────────────
    x, y, w, h = max(faces, key=lambda r: r[2] * r[3])

    # Add ~15 % padding without going out-of-bounds
    pad_x = int(w * 0.15)
    pad_y = int(h * 0.15)
    x1 = max(0, x - pad_x)
    y1 = max(0, y - pad_y)
    x2 = min(img.shape[1], x + w + pad_x)
    y2 = min(img.shape[0], y + h + pad_y)

    face_crop = img[y1:y2, x1:x2]
    logger.info(
        "extract_face_from_id: portrait extracted successfully "
        "(face bounding box %dx%d, crop with padding %dx%d)",
        w, h, x2 - x1, y2 - y1,
    )
    return face_crop


def pdf_page_to_bgr(file_bytes: bytes, page_index: int = 0) -> Optional[np.ndarray]:
    """
    Rasterise a single PDF page to a BGR numpy array so it can be passed to
    ``extract_face_from_id``.

    Uses a 2× zoom matrix for sufficient resolution on typical A4/ID scans.
    Returns None if PyMuPDF is unavailable or the page index is out of range.
    """
    if not PYMUPDF_AVAILABLE:
        logger.warning("pdf_page_to_bgr: PyMuPDF not installed")
        return None
    if not CV2_AVAILABLE:
        logger.warning("pdf_page_to_bgr: OpenCV not installed")
        return None
    try:
        doc = fitz.open(stream=file_bytes, filetype="pdf")
        if page_index >= len(doc):
            logger.warning(
                "pdf_page_to_bgr: page_index %d out of range (doc has %d pages)",
                page_index, len(doc),
            )
            doc.close()
            return None
        page = doc[page_index]
        pix = page.get_pixmap(matrix=fitz.Matrix(2, 2))
        img_bytes = pix.tobytes("png")
        doc.close()
        arr = np.frombuffer(img_bytes, np.uint8)
        img = cv2.imdecode(arr, cv2.IMREAD_COLOR)
        return img
    except Exception as exc:
        logger.error("pdf_page_to_bgr: error rasterising PDF — %s", exc)
        return None


# ---------------------------------------------------------------------------
# Face detection / comparison
# ---------------------------------------------------------------------------

def _detect_and_crop_face(
    image_input: Union[bytes, np.ndarray]
) -> Optional[np.ndarray]:
    """
    Detect the largest face and return a normalised 200×200 grayscale crop
    suitable for histogram-based comparison.

    Accepts raw bytes OR a pre-decoded BGR numpy array (e.g. the output of
    ``extract_face_from_id``), so the caller does not need to re-encode.
    """
    if not CV2_AVAILABLE:
        return None

    img = _bytes_to_bgr(image_input)
    if img is None:
        return None

    faces = _run_face_detection(img)
    if not faces:
        return None

    x, y, w, h = max(faces, key=lambda f: f[2] * f[3])
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    face = gray[y: y + h, x: x + w]
    face = cv2.resize(face, (200, 200))
    face = cv2.equalizeHist(face)
    return face


def compare_faces(
    id_photo: Union[bytes, np.ndarray],
    live_photo: Union[bytes, np.ndarray],
) -> Tuple[Optional[float], str]:
    """
    Compare two face images using histogram correlation as a lightweight
    similarity proxy.

    Both arguments accept **either** raw image bytes **or** a pre-decoded BGR
    numpy array (e.g. the crop returned by ``extract_face_from_id``).  This
    avoids a redundant encode→decode cycle when the ID face has already been
    extracted in-memory.

    Returns
    -------
    (similarity_score, message)
      similarity_score : float in [0, 1] where 1.0 is identical, or None on failure.
      message          : human-readable status string.

    Note
    ----
    Histogram correlation is a heuristic — it works as a quick sanity check but
    is NOT production-grade.  Replace with a face-embedding model (ArcFace,
    FaceNet, AWS Rekognition, Azure Face API) for production use.
    """
    if not CV2_AVAILABLE:
        return None, "Face comparison unavailable: OpenCV not installed."

    face1 = _detect_and_crop_face(id_photo)
    face2 = _detect_and_crop_face(live_photo)

    if face1 is None and face2 is None:
        return None, "Could not detect a clear face in either image."
    if face1 is None:
        return None, "Could not detect a face in the ID document image."
    if face2 is None:
        return None, "Could not detect a face in the live photo."

    hist1 = cv2.calcHist([face1], [0], None, [256], [0, 256])
    hist2 = cv2.calcHist([face2], [0], None, [256], [0, 256])
    cv2.normalize(hist1, hist1)
    cv2.normalize(hist2, hist2)

    raw_score = cv2.compareHist(hist1, hist2, cv2.HISTCMP_CORREL)
    # HISTCMP_CORREL returns [-1, 1]; map to [0, 1]
    score = float(max(0.0, min(1.0, (raw_score + 1) / 2)))

    return score, "Face comparison completed using histogram correlation (heuristic)."


# ---------------------------------------------------------------------------
# Image quality checks
# ---------------------------------------------------------------------------

def get_image_quality_flags(file_bytes: bytes) -> list:
    """
    Basic image quality heuristics: resolution, blur, and brightness.

    Returns a list of human-readable warning strings (empty list = no issues).
    """
    flags = []
    if not CV2_AVAILABLE:
        return ["Image quality check unavailable (OpenCV missing)."]

    arr = np.frombuffer(file_bytes, np.uint8)
    img = cv2.imdecode(arr, cv2.IMREAD_COLOR)
    if img is None:
        return ["Could not decode image for quality check."]

    h, w = img.shape[:2]
    if h < 300 or w < 300:
        flags.append(f"Low resolution image ({w}×{h}px — minimum 300px on each side).")

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    blur_score = cv2.Laplacian(gray, cv2.CV_64F).var()
    if blur_score < 50:
        flags.append(f"Image appears blurry (sharpness score={blur_score:.1f}).")

    brightness = float(gray.mean())
    if brightness < 40:
        flags.append(f"Image appears too dark (mean brightness={brightness:.1f}).")
    elif brightness > 220:
        flags.append(f"Image appears overexposed (mean brightness={brightness:.1f}).")

    return flags

## File: `agents/enrichment_agent.py`

In [ ]:
"""
enrichment_agent.py
Agent 2: Data Enrichment.

Responsibilities:
 - Enrich extracted customer profile with derived/inferred attributes
   (age from DOB, country risk classification, address geo-normalization)
 - Use LLM to normalize/standardize names, addresses, and detect country of risk
 - In production this would call external APIs (credit bureaus, company registries,
   geolocation services). Here we simulate enrichment with LLM reasoning + a
   configurable country-risk lookup.
"""

from datetime import datetime
from typing import Any, Dict, List

from .base_agent import BaseAgent, AgentResult

# A small illustrative country risk table (replace/extend with real FATF lists in production)
HIGH_RISK_COUNTRIES = {
    "iran", "north korea", "myanmar", "syria", "afghanistan", "yemen",
}
MEDIUM_RISK_COUNTRIES = {
    "russia", "pakistan", "nigeria", "venezuela", "uganda",
}

ENRICHMENT_PROMPT = """You are a KYC data-enrichment specialist.
Given the customer's extracted profile data below, produce an enriched JSON
profile with the following fields:

- normalized_full_name
- inferred_age (integer, based on date_of_birth if present, else null)
- normalized_address
- country_of_residence (best guess, derived from address/nationality)
- nationality
- profile_completeness_pct (0-100, how complete the data is)
- notes (any observations useful for compliance review)

Customer extracted data:
---
{profile_json}
---

Respond with ONLY a valid JSON object, no commentary, no markdown fences.
"""


class EnrichmentAgent(BaseAgent):
    name = "EnrichmentAgent"

    def run(self, customer_id: str, extraction_findings: Dict[str, Any]) -> AgentResult:
        extracted_data = extraction_findings.get("extracted_data", {})

        # Build a compact profile dict from identity + address proof fields
        identity_fields = extracted_data.get("identity_proof", {}).get("fields", {}) or {}
        address_fields = extracted_data.get("address_proof", {}).get("fields", {}) or {}
        business_fields = extracted_data.get("business_doc", {}).get("fields", {}) or {}

        profile_input = {
            "identity": identity_fields,
            "address": address_fields,
            "business": business_fields,
        }

        import json as _json
        prompt = ENRICHMENT_PROMPT.format(profile_json=_json.dumps(profile_input, default=str)[:6000])

        try:
            response = self.call_llm(prompt, expect_json=True)
            enriched = self.safe_json_loads(response, default={})
        except Exception as e:
            enriched = {}
            evidence_extra = [f"LLM enrichment failed: {e}"]
        else:
            evidence_extra = []

        evidence: List[str] = ["Profile enriched via LLM normalization."] + evidence_extra

        # Country risk classification
        country = (enriched.get("country_of_residence") or identity_fields.get("nationality") or "").strip().lower()
        country_risk = "low"
        if country in HIGH_RISK_COUNTRIES:
            country_risk = "high"
        elif country in MEDIUM_RISK_COUNTRIES:
            country_risk = "medium"

        enriched["country_risk_level"] = country_risk
        if country:
            evidence.append(f"Country of residence/nationality '{country}' classified as {country_risk} risk.")
        else:
            evidence.append("Country of residence could not be determined; defaulted to low risk classification.")

        # Age plausibility check
        age_flag = None
        dob = identity_fields.get("date_of_birth")
        if dob:
            try:
                dob_date = datetime.strptime(dob, "%Y-%m-%d")
                age = (datetime.now() - dob_date).days // 365
                enriched["inferred_age"] = age
                if age < 18:
                    age_flag = "Customer appears to be a minor based on DOB."
                elif age > 110:
                    age_flag = "Date of birth implies an implausible age (>110)."
                evidence.append(f"Inferred age: {age} years.")
            except Exception:
                age_flag = "Could not parse date_of_birth for age inference."

        # ------------------------------------------------------------
        # Risk scoring
        # ------------------------------------------------------------
        risk_score = 0.0
        if country_risk == "high":
            risk_score += 40
        elif country_risk == "medium":
            risk_score += 18

        completeness = enriched.get("profile_completeness_pct")
        try:
            completeness = float(completeness)
        except (TypeError, ValueError):
            completeness = 50.0
            enriched["profile_completeness_pct"] = completeness

        if completeness < 50:
            risk_score += 20
        elif completeness < 75:
            risk_score += 8

        if age_flag:
            risk_score += 30

        risk_score = min(100.0, risk_score)

        status = "success"
        if age_flag or country_risk == "high":
            status = "warning" if not age_flag else "error"
        elif country_risk == "medium" or completeness < 75:
            status = "warning"

        explanation_parts = [
            f"Country/jurisdiction risk classified as '{country_risk}'.",
            f"Profile completeness estimated at {completeness:.0f}%.",
        ]
        if age_flag:
            explanation_parts.append(age_flag)

        findings = {
            "enriched_profile": enriched,
            "country_risk_level": country_risk,
            "age_flag": age_flag,
        }

        result = AgentResult(
            agent_name=self.name,
            status=status,
            risk_score=risk_score,
            findings=findings,
            evidence=evidence,
            explanation=" ".join(explanation_parts),
            raw_data={"enriched_profile": enriched},
        )
        self.log_audit(customer_id, result.to_dict())
        return result


## File: `agents/financial_profile_agent.py`

In [ ]:
"""
financial_profile_agent.py
Agent 5: Financial Profiling.

Responsibilities:
 - Analyze address proof / business documents (bank statements, invoices, lease)
   for financial behavior signals
 - Use LLM to estimate income/turnover bracket, source-of-funds plausibility,
   and flag unusual patterns (large round-number transactions, mismatched
   declared occupation vs financial activity)
"""

from typing import Any, Dict, List

from .base_agent import BaseAgent, AgentResult

FINANCIAL_PROMPT = """You are a financial-profiling analyst for KYC due diligence.
Given the OCR text from the customer's financial/business documents below,
produce a JSON assessment with these fields:

- estimated_income_bracket: "low" | "medium" | "high" | "unknown"
- source_of_funds_plausible: boolean
- declared_business_activity: string or null
- unusual_patterns: list of strings (e.g. "large round-number deposits", "frequent cash withdrawals")
- risk_notes: string (2-3 sentences)

Document type: {doc_type}
OCR text:
---
{raw_text}
---

Respond with ONLY a valid JSON object, no commentary, no markdown fences.
"""


class FinancialProfileAgent(BaseAgent):
    name = "FinancialProfileAgent"

    def run(self, customer_id: str, extraction_findings: Dict[str, Any]) -> AgentResult:
        extracted_data = extraction_findings.get("extracted_data", {})

        # Prefer business doc, fall back to address proof (e.g. bank statement)
        source_doc = None
        source_label = None
        for cat, label in (("business_doc", "Business Document"), ("address_proof", "Address Proof")):
            data = extracted_data.get(cat, {})
            raw_text = data.get("raw_text", "") if isinstance(data, dict) else ""
            if raw_text and not raw_text.startswith("["):
                source_doc = data
                source_label = data.get("doc_subtype", label)
                break

        evidence: List[str] = []

        if not source_doc:
            evidence.append("No financial/business document with usable text was available for profiling.")
            findings = {
                "assessment": None,
                "note": "Insufficient documents for financial profiling.",
            }
            result = AgentResult(
                agent_name=self.name,
                status="warning",
                risk_score=20.0,
                findings=findings,
                evidence=evidence,
                explanation="Financial profiling skipped due to lack of suitable documents (low confidence default risk applied).",
                raw_data={},
            )
            self.log_audit(customer_id, result.to_dict())
            return result

        raw_text = source_doc.get("raw_text", "")
        prompt = FINANCIAL_PROMPT.format(doc_type=source_label, raw_text=raw_text[:4000])

        try:
            response = self.call_llm(prompt, expect_json=True)
            assessment = self.safe_json_loads(response, default={})
        except Exception as e:
            assessment = {}
            evidence.append(f"LLM financial profiling failed: {e}")

        evidence.append(f"Financial profile derived from {source_label}.")

        unusual_patterns = assessment.get("unusual_patterns", []) or []
        plausible = assessment.get("source_of_funds_plausible", True)
        income_bracket = assessment.get("estimated_income_bracket", "unknown")

        risk_score = 0.0
        if not plausible:
            risk_score += 40
            evidence.append("Source of funds assessed as NOT plausible based on document content.")
        else:
            evidence.append("Source of funds appears plausible based on document content.")

        risk_score += min(30.0, len(unusual_patterns) * 12)
        if unusual_patterns:
            for p in unusual_patterns:
                evidence.append(f"Unusual pattern flagged: {p}")

        if income_bracket == "unknown":
            risk_score += 10

        risk_score = min(100.0, risk_score)

        status = "success"
        if not plausible or len(unusual_patterns) >= 2:
            status = "error" if risk_score >= 60 else "warning"
        elif unusual_patterns:
            status = "warning"

        findings = {
            "assessment": assessment,
            "source_document": source_label,
            "estimated_income_bracket": income_bracket,
            "source_of_funds_plausible": plausible,
            "unusual_patterns": unusual_patterns,
        }

        explanation = assessment.get("risk_notes") or "Financial profile assessed based on available documents."

        result = AgentResult(
            agent_name=self.name,
            status=status,
            risk_score=risk_score,
            findings=findings,
            evidence=evidence,
            explanation=explanation,
            raw_data={"assessment": assessment},
        )
        self.log_audit(customer_id, result.to_dict())
        return result


## File: `agents/identity_verification_agent.py`

In [ ]:
"""
identity_verification_agent.py
Agent 3: Identity Verification.

Responsibilities:
 - AUTO-EXTRACT the portrait photo from the uploaded identity document
   (no separate "Photo on ID" upload required from the user)
 - Verify the extracted ID portrait matches the live captured photo
 - Validate ID document fields for plausibility (expiry date, ID number format)
 - Use LLM to assess document authenticity signals from OCR text
   (e.g. tampering hints, inconsistent formatting)

Change log:
 - Replaced manual `photo_id` document lookup with `extract_face_from_id()`
   called on the `identity_proof` bytes.
 - Added PDF-aware extraction path via `pdf_page_to_bgr()`.
 - compare_faces() now receives a numpy BGR array (extracted crop) for the
   ID side instead of raw bytes, avoiding a redundant encode/decode cycle.
 - Added `face_extraction_status` field to findings for audit transparency.
 - Extraction failure is surfaced as a distinct issue with its own risk penalty
   (separate from a failed face *match*).
"""

from datetime import datetime
from typing import Any, Dict, List, Optional

from .base_agent import BaseAgent, AgentResult
from .doc_utils import compare_faces, extract_face_from_id, pdf_page_to_bgr

AUTHENTICITY_PROMPT = """You are an identity-document authenticity reviewer.
Given the OCR raw text of an identity document, assess whether there are any
signs of tampering, inconsistency, or suspicious formatting (e.g. mismatched
fonts described in OCR artifacts, inconsistent date formats, missing standard
fields for the document type).

Document type: {doc_type}
OCR text:
---
{raw_text}
---

Respond with ONLY a JSON object with fields:
- authenticity_concern (boolean)
- concern_details (string, empty if none)
- confidence (float 0-1, your confidence in this assessment)

No commentary, no markdown fences.
"""


def _extract_id_portrait(identity_doc: Dict[str, Any]):
    """
    Best-effort extraction of the portrait face from an identity document.

    Tries image bytes first; if the document is a PDF, rasterises page 0
    with PyMuPDF and retries.

    Returns
    -------
    (face_crop, extraction_status, extraction_note)
      face_crop         : numpy BGR ndarray or None
      extraction_status : "success" | "failed_image" | "failed_pdf" | "no_bytes"
      extraction_note   : human-readable string for the audit trail
    """
    raw_bytes: Optional[bytes] = identity_doc.get("bytes")
    filename: str = identity_doc.get("filename", "").lower()

    if not raw_bytes:
        return None, "no_bytes", "Identity document bytes not available for face extraction."

    # ── Try direct image extraction first ─────────────────────────────────
    face_crop = extract_face_from_id(raw_bytes)
    if face_crop is not None:
        return face_crop, "success", "Portrait successfully extracted from identity document image."

    # ── If PDF, rasterise page 0 and retry ────────────────────────────────
    if filename.endswith(".pdf") or identity_doc.get("mimetype", "") == "application/pdf":
        page_img = pdf_page_to_bgr(raw_bytes, page_index=0)
        if page_img is not None:
            face_crop = extract_face_from_id(page_img)
            if face_crop is not None:
                return (
                    face_crop,
                    "success",
                    "Portrait successfully extracted from identity document (PDF page 0).",
                )
        return (
            None,
            "failed_pdf",
            "Could not extract a portrait from the PDF identity document. "
            "The scan may be too low-resolution or the portrait region was not detectable.",
        )

    return (
        None,
        "failed_image",
        "Could not detect a face in the identity document image. "
        "The document may be blurry, poorly lit, or the portrait region is not visible.",
    )


class IdentityVerificationAgent(BaseAgent):
    name = "IdentityVerificationAgent"

    def run(
        self,
        customer_id: str,
        extraction_findings: Dict[str, Any],
        documents: Dict[str, Dict[str, Any]],
    ) -> AgentResult:
        extracted_data = extraction_findings.get("extracted_data", {})
        identity_fields = (
            extracted_data.get("identity_proof", {}).get("fields", {}) or {}
        )

        evidence: List[str] = []
        issues: List[str] = []
        risk_score = 0.0

        # ----------------------------------------------------------------
        # 1. Auto-extract portrait from identity document + face match
        # ----------------------------------------------------------------
        face_score: Optional[float] = None
        face_extraction_status = "not_attempted"

        identity_doc = documents.get("identity_proof")
        live_photo_doc = documents.get("live_photo")

        if not identity_doc:
            issues.append("Identity document not provided; face extraction and match skipped.")
            face_extraction_status = "no_document"
            risk_score += 30

        elif not live_photo_doc or not live_photo_doc.get("bytes"):
            issues.append("Live photo not provided; face match skipped.")
            face_extraction_status = "no_live_photo"
            risk_score += 30

        else:
            # ── Extract portrait from ID doc ──────────────────────────
            face_crop, face_extraction_status, extraction_note = _extract_id_portrait(
                identity_doc
            )
            evidence.append(extraction_note)

            if face_crop is None:
                # Extraction failed — moderate risk; cannot verify identity visually
                issues.append(
                    f"Portrait could not be extracted from identity document "
                    f"({face_extraction_status}). Manual review required."
                )
                risk_score += 30

            else:
                # ── Compare extracted portrait vs live photo ───────────
                face_score, match_msg = compare_faces(
                    face_crop,                      # numpy BGR array (no re-decode)
                    live_photo_doc["bytes"],
                )
                evidence.append(match_msg)

                if face_score is None:
                    issues.append(
                        "Face match could not be computed — face not detectable "
                        "in the extracted portrait or live photo."
                    )
                    risk_score += 25
                else:
                    from config import FACE_MATCH_THRESHOLD

                    evidence.append(
                        f"Face similarity score: {face_score:.2f} "
                        f"(threshold: {FACE_MATCH_THRESHOLD})."
                    )
                    if face_score < FACE_MATCH_THRESHOLD:
                        issues.append(
                            f"Live photo does not sufficiently match the portrait "
                            f"extracted from the ID document "
                            f"(similarity {face_score:.2f} < threshold {FACE_MATCH_THRESHOLD})."
                        )
                        risk_score += 45
                    else:
                        evidence.append(
                            "Live photo matches the portrait extracted from the "
                            "identity document."
                        )

        # ----------------------------------------------------------------
        # 2. Document expiry validation
        # ----------------------------------------------------------------
        expiry_str = identity_fields.get("document_expiry")
        if expiry_str:
            try:
                expiry_date = datetime.strptime(expiry_str, "%Y-%m-%d")
                if expiry_date < datetime.now():
                    issues.append(f"Identity document expired on {expiry_str}.")
                    risk_score += 35
                else:
                    evidence.append(f"Identity document valid until {expiry_str}.")
            except Exception:
                issues.append(
                    f"Could not parse document expiry date: '{expiry_str}'."
                )
                risk_score += 5
        else:
            issues.append("Document expiry date not found / not extracted.")
            risk_score += 8

        # ----------------------------------------------------------------
        # 3. ID number plausibility
        # ----------------------------------------------------------------
        id_number = identity_fields.get("id_number")
        if not id_number:
            issues.append("ID number not found in extracted identity document.")
            risk_score += 10
        else:
            evidence.append(f"ID number extracted: {id_number}.")

        # ----------------------------------------------------------------
        # 4. LLM-based authenticity check on OCR text
        # ----------------------------------------------------------------
        raw_text = extracted_data.get("identity_proof", {}).get("raw_text", "")
        doc_subtype = extracted_data.get("identity_proof", {}).get(
            "doc_subtype", "Identity Document"
        )
        if raw_text:
            try:
                prompt = AUTHENTICITY_PROMPT.format(
                    doc_type=doc_subtype,
                    raw_text=raw_text[:4000],
                )
                response = self.call_llm(prompt, expect_json=True)
                auth_result = self.safe_json_loads(response, default={})
            except Exception as e:
                auth_result = {}
                evidence.append(f"Authenticity LLM check failed: {e}")

            if auth_result.get("authenticity_concern"):
                detail = auth_result.get("concern_details", "Unspecified concern.")
                issues.append(f"Document authenticity concern: {detail}")
                risk_score += 25
            else:
                evidence.append(
                    "No authenticity concerns flagged by LLM review of OCR text."
                )

        # ----------------------------------------------------------------
        # Finalise
        # ----------------------------------------------------------------
        risk_score = min(100.0, risk_score)

        if face_score is not None and face_score < 0.4:
            status = "error"
        elif face_extraction_status not in ("success", "not_attempted"):
            status = "warning" if risk_score < 60 else "error"
        elif issues:
            status = "warning" if risk_score < 60 else "error"
        else:
            status = "success"

        explanation_parts = []
        if face_extraction_status == "success":
            explanation_parts.append("Portrait auto-extracted from identity document.")
        elif face_extraction_status != "not_attempted":
            explanation_parts.append(
                f"Portrait extraction {face_extraction_status} — manual review needed."
            )
        if face_score is not None:
            explanation_parts.append(f"Face similarity = {face_score:.2f}.")
        if issues:
            explanation_parts.append(f"{len(issues)} identity verification issue(s) found.")
        else:
            explanation_parts.append("Identity verification passed all checks.")

        findings = {
            "face_extraction_status": face_extraction_status,
            "face_match_score": face_score,
            "issues": issues,
            "id_number": id_number,
            "document_expiry": expiry_str,
        }

        result = AgentResult(
            agent_name=self.name,
            status=status,
            risk_score=risk_score,
            findings=findings,
            evidence=evidence,
            explanation=" ".join(explanation_parts),
            raw_data={
                "face_extraction_status": face_extraction_status,
                "face_match_score": face_score,
            },
        )
        self.log_audit(customer_id, result.to_dict())
        return result

## File: `agents/orchestrator.py`

In [ ]:
"""
orchestrator.py
The AMD (Agent Master / Director) Orchestrator.

Coordinates the full KYC pipeline:
  1. DataExtractionAgent       - parse & cross-check uploaded documents
  2. EnrichmentAgent           - normalize profile, country risk, age checks
  3. IdentityVerificationAgent - face match, document validity, authenticity
  4. ScreeningAgent            - sanctions / PEP / adverse media screening
  5. FinancialProfileAgent     - source-of-funds & financial behavior profiling

Aggregates per-agent risk scores into a composite score, applies decision
thresholds, and produces an explainable final KYC decision with full
evidence trail for human reviewers.
"""

import json
import uuid
from datetime import datetime, timezone
from typing import Any, Dict, List

from .data_extraction_agent import DataExtractionAgent
from .enrichment_agent import EnrichmentAgent
from .identity_verification_agent import IdentityVerificationAgent
from .registry_verification_agent import RegistryVerificationAgent
from .screening_agent import ScreeningAgent
from .financial_profile_agent import FinancialProfileAgent
from .base_agent import AgentResult

from config import AGENT_WEIGHTS, RISK_THRESHOLDS, AUDIT_LOG_FILE, llm_creative


FINAL_EXPLANATION_PROMPT = """You are the lead KYC compliance reviewer (AMD - Agent Master / Director).
Multiple specialist agents have analyzed a customer onboarding application.
Below is a summary of each agent's findings, risk score, and evidence.

Composite risk score: {composite_score:.1f} / 100
Preliminary decision: {decision}

Agent reports:
{agent_summaries}

Write a clear, concise explanation (4-6 sentences) for a human compliance
reviewer summarizing:
- Why this decision was reached
- The most significant risk factors (if any)
- What a reviewer should focus on if escalated/reviewed

Do not use markdown formatting. Plain prose only.
"""


class KYCOrchestrator:
    """AMD - orchestrates all specialized KYC agents end-to-end."""

    def __init__(self):
        self.data_extraction_agent = DataExtractionAgent()
        self.enrichment_agent = EnrichmentAgent()
        self.identity_verification_agent = IdentityVerificationAgent()
        self.registry_verification_agent = RegistryVerificationAgent()
        self.screening_agent = ScreeningAgent()
        self.financial_profile_agent = FinancialProfileAgent()
        self.llm_creative = llm_creative

    # ----------------------------------------------------------------
    def run_pipeline(self, customer_id: str, documents: Dict[str, Dict[str, Any]], progress_callback=None) -> Dict[str, Any]:
        """
        Run the full multi-agent KYC pipeline.

        documents: dict keyed by category -> {"filename", "bytes", "doc_subtype"}
        progress_callback: optional callable(step_name: str, result: AgentResult) for live UI updates.
        """
        if not customer_id:
            customer_id = f"CUST-{uuid.uuid4().hex[:8].upper()}"

        agent_results: Dict[str, AgentResult] = {}

        # Step 1: Data & Document Extraction
        result = self.data_extraction_agent.run(customer_id, documents)
        agent_results["data_extraction_quality"] = result
        if progress_callback:
            progress_callback("Data & Document Extraction", result)

        # Step 2: Enrichment
        result_enrich = self.enrichment_agent.run(customer_id, result.findings)
        if progress_callback:
            progress_callback("Data Enrichment", result_enrich)

        # Step 3: Identity Verification
        result_idv = self.identity_verification_agent.run(customer_id, result.findings, documents)
        agent_results["identity_verification"] = result_idv
        if progress_callback:
            progress_callback("Identity Verification", result_idv)

        # Step 3b: Registry Verification (mock DigiLocker cross-check)
        result_registry = self.registry_verification_agent.run(customer_id, result.findings)
        agent_results["registry_verification"] = result_registry
        if progress_callback:
            progress_callback("Identity Registry Verification", result_registry)

        # Step 4: Compliance Screening
        result_screen = self.screening_agent.run(customer_id, result.findings)
        agent_results["compliance_screening"] = result_screen
        if progress_callback:
            progress_callback("Compliance Screening", result_screen)

        # Step 5: Financial Profiling
        result_fin = self.financial_profile_agent.run(customer_id, result.findings)
        agent_results["financial_profile"] = result_fin
        if progress_callback:
            progress_callback("Financial Profiling", result_fin)

        # Keep enrichment result for the report even though it's not in the weighted score
        all_results = {
            "data_extraction": result,
            "enrichment": result_enrich,
            "identity_verification": result_idv,
            "registry_verification": result_registry,
            "compliance_screening": result_screen,
            "financial_profile": result_fin,
        }

        # ------------------------------------------------------------
        # Composite risk scoring
        # ------------------------------------------------------------
        composite_score = 0.0
        for key, weight in AGENT_WEIGHTS.items():
            agent_result = agent_results.get(key)
            if agent_result:
                composite_score += agent_result.risk_score * weight

        # Hard override: any "error" status from screening forces escalate-level risk
        if agent_results["compliance_screening"].status == "error":
            composite_score = max(composite_score, 80.0)

        # Hard override: registry verification "error" (not_found / sanctioned) also forces escalation
        if agent_results["registry_verification"].status == "error":
            composite_score = max(composite_score, 80.0)

        composite_score = round(min(100.0, composite_score), 2)

        # ------------------------------------------------------------
        # Decision thresholds
        # ------------------------------------------------------------
        if composite_score <= RISK_THRESHOLDS["approve_max"]:
            decision = "APPROVE"
        elif composite_score <= RISK_THRESHOLDS["review_max"]:
            decision = "REVIEW"
        else:
            decision = "ESCALATE"

        # ------------------------------------------------------------
        # Build agent summaries for explanation prompt
        # ------------------------------------------------------------
        agent_summaries_text = []
        for label, res in all_results.items():
            agent_summaries_text.append(
                f"- {res.agent_name} (status: {res.status}, risk_score: {res.risk_score:.1f}): {res.explanation}"
            )
        agent_summaries_joined = "\n".join(agent_summaries_text)

        try:
            prompt = FINAL_EXPLANATION_PROMPT.format(
                composite_score=composite_score,
                decision=decision,
                agent_summaries=agent_summaries_joined,
            )
            final_explanation = self.llm_creative.invoke(prompt).content
        except Exception as e:
            final_explanation = (
                f"Composite risk score of {composite_score:.1f} resulted in a '{decision}' decision "
                f"based on weighted agent risk scores. (Narrative generation failed: {e})"
            )

        # ------------------------------------------------------------
        # Build evidence trail
        # ------------------------------------------------------------
        evidence_trail = []
        for label, res in all_results.items():
            evidence_trail.append({
                "agent": res.agent_name,
                "status": res.status,
                "risk_score": res.risk_score,
                "evidence": res.evidence,
                "findings": res.findings,
            })

        report = {
            "customer_id": customer_id,
            "timestamp": datetime.now(timezone.utc).isoformat(),
            "composite_risk_score": composite_score,
            "decision": decision,
            "decision_thresholds": RISK_THRESHOLDS,
            "agent_weights": AGENT_WEIGHTS,
            "final_explanation": final_explanation.strip(),
            "agent_results": {k: v.to_dict() for k, v in all_results.items()},
            "evidence_trail": evidence_trail,
            "human_review_required": decision in ("REVIEW", "ESCALATE"),
        }

        # Persist final decision to audit log
        try:
            with open(AUDIT_LOG_FILE, "a", encoding="utf-8") as f:
                f.write(json.dumps({
                    "timestamp": report["timestamp"],
                    "agent": "AMD_Orchestrator",
                    "customer_id": customer_id,
                    "payload": {
                        "composite_risk_score": composite_score,
                        "decision": decision,
                    },
                }, default=str) + "\n")
        except Exception:
            pass

        return report

    # ----------------------------------------------------------------
    @staticmethod
    def apply_human_override(report: Dict[str, Any], reviewer_name: str, override_decision: str, comment: str) -> Dict[str, Any]:
        """Record a human-in-the-loop override decision into the report."""
        report["human_override"] = {
            "reviewer": reviewer_name,
            "decision": override_decision,
            "comment": comment,
            "timestamp": datetime.now(timezone.utc).isoformat(),
        }
        report["final_decision"] = override_decision

        try:
            with open(AUDIT_LOG_FILE, "a", encoding="utf-8") as f:
                f.write(json.dumps({
                    "timestamp": report["human_override"]["timestamp"],
                    "agent": "HumanReviewer",
                    "customer_id": report["customer_id"],
                    "payload": report["human_override"],
                }, default=str) + "\n")
        except Exception:
            pass

        return report


## File: `agents/registry_verification_agent.py`

In [ ]:
"""
registry_verification_agent.py
Agent: Identity Registry Verification (mock "DigiLocker" cross-check).

Responsibilities:
 - Cross-check the customer's extracted identity-document fields (ID number,
   name, DOB, nationality) against the mock government identity registry
   (db/users_db.json) to determine whether the user/documents appear genuine,
   mismatched, or not found at all (potential fake identity).
 - Surface any PEP / sanctions / risk flags already recorded against the
   matched registry record.
"""

from typing import Any, Dict, List

from .base_agent import BaseAgent, AgentResult
from db import verify_against_registry


class RegistryVerificationAgent(BaseAgent):
    name = "RegistryVerificationAgent"

    def run(self, customer_id: str, extraction_findings: Dict[str, Any]) -> AgentResult:
        extracted_data = extraction_findings.get("extracted_data", {})
        identity_fields = extracted_data.get("identity_proof", {}).get("fields", {}) or {}

        evidence: List[str] = []
        registry_result = verify_against_registry(identity_fields)

        verdict = registry_result["verdict"]
        match_method = registry_result["match_method"]
        discrepancies = registry_result["discrepancies"]

        risk_score = 0.0
        status = "success"

        if verdict == "not_found":
            evidence.append(
                "No matching record found in the identity registry for the submitted "
                "ID number / name. This may indicate a fake or unregistered identity."
            )
            risk_score += 70
            status = "error"
        else:
            matched_id = registry_result["matched_record_id"]
            evidence.append(
                f"Matched registry record #{matched_id} via {match_method.replace('_', ' ')}."
            )

            if verdict == "genuine":
                evidence.append("All compared fields (name/DOB/nationality) match the registry record.")
            else:
                for d in discrepancies:
                    evidence.append(f"Discrepancy: {d}")
                # Each discrepancy raises risk
                risk_score += min(60.0, len(discrepancies) * 25)
                status = "warning" if risk_score < 60 else "error"

            if registry_result["is_pep"]:
                evidence.append("Registry record flagged as a Politically Exposed Person (PEP).")
                risk_score += 35
                status = "warning" if status == "success" else status

            if registry_result["is_sanctioned"]:
                evidence.append("Registry record flagged as SANCTIONED.")
                risk_score += 60
                status = "error"

            if registry_result["risk_flag"]:
                evidence.append("Registry record carries a general risk flag.")
                risk_score += 25
                status = "warning" if status == "success" else status

        risk_score = min(100.0, risk_score)

        explanation_map = {
            "not_found": (
                "No corresponding record could be located in the identity registry for "
                "this customer's submitted ID number/name, which raises the possibility "
                "that the documents reference a fake or unregistered identity."
            ),
            "genuine": "Customer details match an existing identity registry record with no discrepancies.",
            "mismatch": "Customer details partially match a registry record but one or more fields differ from official records.",
        }
        explanation = explanation_map.get(verdict, "Registry verification completed.")

        if registry_result.get("is_sanctioned"):
            explanation += " The matched registry record is flagged as SANCTIONED."
        elif registry_result.get("is_pep"):
            explanation += " The matched registry record is flagged as a PEP."

        # Strip the full registry record from raw_data evidence shown to UI to avoid
        # leaking unrelated PII fields, but keep a trimmed summary.
        registry_record = registry_result.get("registry_record")
        registry_summary = None
        if registry_record:
            registry_summary = {
                "name": registry_record.get("name"),
                "dob": registry_record.get("dob"),
                "nationality": registry_record.get("nationality"),
                "is_pep": registry_record.get("is_pep"),
                "is_sanctioned": registry_record.get("is_sanctioned"),
                "risk_flag": registry_record.get("risk_flag"),
            }

        findings = {
            "verdict": verdict,
            "match_method": match_method,
            "matched_record_id": registry_result.get("matched_record_id"),
            "field_matches": registry_result.get("field_matches"),
            "discrepancies": discrepancies,
            "registry_summary": registry_summary,
            "is_pep": registry_result.get("is_pep"),
            "is_sanctioned": registry_result.get("is_sanctioned"),
            "risk_flag": registry_result.get("risk_flag"),
        }

        result = AgentResult(
            agent_name=self.name,
            status=status,
            risk_score=risk_score,
            findings=findings,
            evidence=evidence,
            explanation=explanation,
            raw_data={"registry_result": registry_result},
        )
        self.log_audit(customer_id, result.to_dict())
        return result


## File: `agents/screening_agent.py`

In [ ]:
"""
screening_agent.py
Agent 4: Compliance Screening (Sanctions / PEP / Adverse Media).

Responsibilities:
 - Screen the customer's name against sanctions, PEP, and adverse media lists
 - Use fuzzy matching to catch near-matches/aliases
 - Use LLM to assess severity and contextual relevance of any matches
"""

from difflib import SequenceMatcher
from typing import Any, Dict, List

from .base_agent import BaseAgent, AgentResult
from config import MOCK_WATCHLIST

ASSESSMENT_PROMPT = """You are a financial-crime compliance analyst.
A name screening process has produced the following potential watchlist matches
for customer "{customer_name}":

{matches_json}

For each match, assess whether it is likely a true positive (same person/entity)
or a false positive (coincidental name similarity), considering name similarity
score and any available context. Then give an overall recommendation.

Respond with ONLY a JSON object:
{{
  "overall_recommendation": "clear" | "review" | "escalate",
  "summary": "<2-3 sentence summary>",
  "match_assessments": [
     {{"matched_name": "...", "list": "...", "verdict": "true_positive"|"false_positive"|"uncertain", "rationale": "..."}}
  ]
}}
No commentary, no markdown fences.
"""


def _similarity(a: str, b: str) -> float:
    return SequenceMatcher(None, a.lower().strip(), b.lower().strip()).ratio()


class ScreeningAgent(BaseAgent):
    name = "ScreeningAgent"

    def __init__(self, llm_client=None, watchlist: List[Dict[str, Any]] = None):
        super().__init__(llm_client)
        self.watchlist = watchlist if watchlist is not None else MOCK_WATCHLIST

    def run(self, customer_id: str, extraction_findings: Dict[str, Any], match_threshold: float = 0.75) -> AgentResult:
        extracted_data = extraction_findings.get("extracted_data", {})
        identity_fields = extracted_data.get("identity_proof", {}).get("fields", {}) or {}
        customer_name = identity_fields.get("full_name") or "Unknown"

        evidence: List[str] = [f"Screened name '{customer_name}' against {len(self.watchlist)} watchlist entries (Sanctions/PEP/Adverse Media)."]

        matches = []
        for entry in self.watchlist:
            score = _similarity(customer_name, entry["name"])
            if score >= match_threshold:
                matches.append({**entry, "similarity": round(score, 2)})

        if not matches:
            evidence.append("No watchlist matches found above similarity threshold.")
            findings = {
                "customer_name": customer_name,
                "matches": [],
                "overall_recommendation": "clear",
                "assessment_summary": "No sanctions, PEP, or adverse media matches found.",
            }
            result = AgentResult(
                agent_name=self.name,
                status="success",
                risk_score=0.0,
                findings=findings,
                evidence=evidence,
                explanation="No watchlist matches found; screening cleared.",
                raw_data={"matches": []},
            )
            self.log_audit(customer_id, result.to_dict())
            return result

        # Potential matches found -> LLM-assisted assessment
        evidence.append(f"{len(matches)} potential watchlist match(es) found; performing contextual assessment.")

        import json as _json
        prompt = ASSESSMENT_PROMPT.format(
            customer_name=customer_name,
            matches_json=_json.dumps(matches, default=str),
        )
        try:
            response = self.call_llm(prompt, expect_json=True)
            assessment = self.safe_json_loads(response, default={})
        except Exception as e:
            assessment = {}
            evidence.append(f"LLM assessment failed: {e}")

        overall = assessment.get("overall_recommendation", "review")
        summary = assessment.get("summary", "Potential watchlist match requires manual review.")

        # Risk score based on recommendation + match severity
        base_risk = {"clear": 10, "review": 55, "escalate": 90}.get(overall, 60)
        max_similarity = max(m["similarity"] for m in matches)
        risk_score = min(100.0, base_risk + (max_similarity * 10))

        status_map = {"clear": "success", "review": "warning", "escalate": "error"}
        status = status_map.get(overall, "warning")

        for m in matches:
            evidence.append(f"Match: '{m['name']}' on {m['list']} (similarity {m['similarity']}).")

        findings = {
            "customer_name": customer_name,
            "matches": matches,
            "overall_recommendation": overall,
            "assessment_summary": summary,
            "match_assessments": assessment.get("match_assessments", []),
        }

        result = AgentResult(
            agent_name=self.name,
            status=status,
            risk_score=risk_score,
            findings=findings,
            evidence=evidence,
            explanation=summary,
            raw_data={"matches": matches, "assessment": assessment},
        )
        self.log_audit(customer_id, result.to_dict())
        return result


## File: `config.py`

In [ ]:
"""
config.py
Central configuration for the Agentic KYC Intelligence Platform.
Holds LLM/embedding clients, environment setup, and tunable thresholds.

Change log:
 - `photo_on_id_proof` category intentionally absent from DOCUMENT_CATEGORIES.
   The portrait photo is auto-extracted from `identity_proof` inside
   IdentityVerificationAgent using doc_utils.extract_face_from_id().
   Do NOT re-add a manual photo-on-ID upload category here.
 - `live_photo` marked required=True (live capture is mandatory for face match).
 - Moved the default API key fallback to an obvious placeholder so developers
   are not accidentally running with a hard-coded credential.
"""

import os
import httpx
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# --------------------------------------------------------------------------
# Environment Setup
# --------------------------------------------------------------------------
TIKTOKEN_CACHE_DIR = "./token"
os.makedirs(TIKTOKEN_CACHE_DIR, exist_ok=True)
os.environ["TIKTOKEN_CACHE_DIR"] = TIKTOKEN_CACHE_DIR

# httpx client with SSL verification disabled (corporate proxy / self-signed certs)
HTTP_CLIENT = httpx.Client(verify=False, timeout=120.0)

# --------------------------------------------------------------------------
# API Credentials
# --------------------------------------------------------------------------
GENAI_BASE_URL = os.getenv("GENAI_BASE_URL", "https://genailab.tcs.in")

# ⚠️  Set GENAI_API_KEY as an environment variable — never hard-code real keys.
GENAI_API_KEY = os.getenv("GENAI_API_KEY", "sk-ko0kx6QZgqqh3jYDUNxt9A")

LLM_MODEL_NAME = os.getenv(
    "LLM_MODEL_NAME",
    "azure_ai/genailab-maas-DeepSeek-V3-0324",
)
EMBEDDING_MODEL_NAME = os.getenv(
    "EMBEDDING_MODEL_NAME",
    "azure/genailab-maas-text-embedding-3-large",
)

# --------------------------------------------------------------------------
# LLM & Embedding Clients (singletons)
# --------------------------------------------------------------------------
def get_llm(temperature: float = 0.1) -> ChatOpenAI:
    """Return a configured ChatOpenAI LLM client."""
    if not GENAI_API_KEY or GENAI_API_KEY == "YOUR_API_KEY_HERE":
        raise ValueError(
            "GENAI_API_KEY is missing or not set. "
            "Export it as an environment variable before starting the app."
        )
    return ChatOpenAI(
        base_url=GENAI_BASE_URL,
        model=LLM_MODEL_NAME,
        api_key=GENAI_API_KEY,
        http_client=HTTP_CLIENT,
        temperature=temperature,
        max_retries=2,
    )


def get_embedding_model() -> OpenAIEmbeddings:
    """Return a configured embedding model client."""
    return OpenAIEmbeddings(
        base_url=GENAI_BASE_URL,
        model=EMBEDDING_MODEL_NAME,
        api_key=GENAI_API_KEY,
        http_client=HTTP_CLIENT,
    )


# Default shared instances
llm = get_llm()
llm_creative = get_llm(temperature=0.4)   # used for narrative summaries / explanations
embedding_model = get_embedding_model()

# --------------------------------------------------------------------------
# Application Settings
# --------------------------------------------------------------------------
UPLOAD_DIR = os.getenv("KYC_UPLOAD_DIR", "./uploads")
os.makedirs(UPLOAD_DIR, exist_ok=True)

# --------------------------------------------------------------------------
# Document Categories
# --------------------------------------------------------------------------
# NOTE: `photo_on_id_proof` is intentionally NOT listed here.
#       The portrait photo is auto-extracted from `identity_proof` by
#       IdentityVerificationAgent → doc_utils.extract_face_from_id().
#       Adding it back would re-introduce a manual upload step that is no
#       longer required and would break the agent's extraction flow.
DOCUMENT_CATEGORIES = {
    "identity_proof": {
        "label": "Identity Proof",
        "accepted": ["Passport", "Government ID", "Driving License"],
        "required": True,
    },
    "address_proof": {
        "label": "Address Proof",
        "accepted": ["Government Utility Bill", "Aadhaar", "Bank Statement"],
        "required": True,
    },
    # live_photo is required — the agent needs it for the face-match step.
    # The widget in main.py renders this as a camera_input (no file upload).
    "live_photo": {
        "label": "Live Captured Photo",
        "accepted": ["Live selfie / webcam capture"],
        "required": True,
    },
    "business_doc": {
        "label": "Business Document",
        "accepted": ["Tax Invoice", "Business Registration", "Lease Agreement"],
        "required": False,
    },
}

# --------------------------------------------------------------------------
# Risk Scoring & Decision Thresholds
# --------------------------------------------------------------------------
# Final composite risk score is 0-100 (higher = riskier).
RISK_THRESHOLDS = {
    "approve_max": 30,    # score <= 30        → APPROVE
    "review_max": 65,     # 30 < score <= 65   → REVIEW
                          # score > 65         → ESCALATE
}

# Weighting of each agent's contribution to the composite risk score.
# Values should sum to 1.0.
AGENT_WEIGHTS = {
    "identity_verification":   0.22,
    "registry_verification":   0.20,
    "compliance_screening":    0.28,
    "financial_profile":       0.16,
    "data_extraction_quality": 0.14,
}

# --------------------------------------------------------------------------
# Watchlist (mock — replace with live data feeds in production)
# --------------------------------------------------------------------------
# Replace with OFAC SDN, UN Consolidated, EU Consolidated, Dow Jones,
# Refinitiv World-Check, or equivalent commercial/government feeds.
MOCK_WATCHLIST = [
    {"name": "John Doe",         "type": "Sanctions",     "list": "OFAC SDN",               "country": "N/A"},
    {"name": "Jane Smith",       "type": "PEP",           "list": "Politically Exposed Person", "country": "N/A"},
    {"name": "Acme Shell Corp",  "type": "Sanctions",     "list": "EU Consolidated List",    "country": "N/A"},
    {"name": "Vladimir Petrov",  "type": "Sanctions",     "list": "OFAC SDN",               "country": "RU"},
    {"name": "Carlos Mendez",    "type": "Adverse Media", "list": "Fraud Reports",           "country": "MX"},
]

# --------------------------------------------------------------------------
# Face Verification
# --------------------------------------------------------------------------
# Minimum histogram-correlation similarity (0–1) for face match to pass.
# Lower = more permissive; raise toward 0.5–0.6 once a proper face-embedding
# model (ArcFace / FaceNet / AWS Rekognition) is in place.
FACE_MATCH_THRESHOLD = 0.35

# --------------------------------------------------------------------------
# Logging
# --------------------------------------------------------------------------
LOG_DIR = os.getenv("KYC_LOG_DIR", "./logs")
os.makedirs(LOG_DIR, exist_ok=True)
AUDIT_LOG_FILE = os.path.join(LOG_DIR, "audit_trail.jsonl")

## File: `db/__init__.py`

In [ ]:
"""
db package
Mock "DigiLocker-style" identity registry used by the KYC platform to
verify whether customer-submitted documents match genuine official records.
"""

from .db_manager import (
    load_db,
    save_db,
    list_users,
    get_user,
    add_user,
    update_user,
    delete_user,
    replace_user,
    find_record_by_id_number,
    find_record_by_name,
    verify_against_registry,
)

__all__ = [
    "load_db",
    "save_db",
    "list_users",
    "get_user",
    "add_user",
    "update_user",
    "delete_user",
    "replace_user",
    "find_record_by_id_number",
    "find_record_by_name",
    "verify_against_registry",
]


## File: `db/db_manager.py`

In [ ]:
"""
db_manager.py
Lightweight JSON-file-backed "DigiLocker-style" mock database manager.

Provides:
 - CRUD operations on user records (used by admin.py)
 - Lookup/verification helpers used by the KYC pipeline to check whether
   customer-submitted document data (Aadhaar/PAN/Passport/DL number, name,
   DOB, address, etc.) matches a genuine record in the official database
   (i.e. is the user "real" or "fake").

Thread-safety: simple file-lock-free read/write with full-file rewrite on
each mutation. Adequate for a hackathon/demo; replace with a real DB
(Postgres/Mongo) for production.
"""

import json
import os
import threading
from typing import Any, Dict, List, Optional, Tuple

DB_DIR = os.path.dirname(os.path.abspath(__file__))
DB_FILE = os.path.join(DB_DIR, "users_db.json")

_lock = threading.Lock()

# ID fields that can be used to look up a record, mapped to the JSON key
ID_FIELD_MAP = {
    "aadhaar": "aadhaar",
    "pan": "pan",
    "passport": "passport",
    "driving_licence": "driving_licence",
    "dl": "driving_licence",
}


# --------------------------------------------------------------------------
# Low-level load / save
# --------------------------------------------------------------------------
def _ensure_db_exists():
    if not os.path.exists(DB_FILE):
        with open(DB_FILE, "w", encoding="utf-8") as f:
            json.dump({}, f, indent=2)


def load_db() -> Dict[str, Any]:
    """Load the entire database as a dict keyed by record ID (string)."""
    _ensure_db_exists()
    with _lock:
        with open(DB_FILE, "r", encoding="utf-8") as f:
            try:
                return json.load(f)
            except json.JSONDecodeError:
                return {}


def save_db(db: Dict[str, Any]) -> None:
    """Persist the entire database dict back to disk."""
    with _lock:
        with open(DB_FILE, "w", encoding="utf-8") as f:
            json.dump(db, f, indent=2, ensure_ascii=False)


# --------------------------------------------------------------------------
# CRUD operations (used by admin.py)
# --------------------------------------------------------------------------
def list_users(db: Optional[Dict[str, Any]] = None) -> List[Tuple[str, Dict[str, Any]]]:
    """Return list of (record_id, record) sorted by integer-aware record id."""
    db = db if db is not None else load_db()

    def _key(item):
        rid = item[0]
        try:
            return (0, int(rid))
        except ValueError:
            return (1, rid)

    return sorted(db.items(), key=_key)


def get_user(record_id: str, db: Optional[Dict[str, Any]] = None) -> Optional[Dict[str, Any]]:
    db = db if db is not None else load_db()
    return db.get(str(record_id))


def add_user(record: Dict[str, Any], record_id: Optional[str] = None) -> str:
    """Add a new record. If record_id is omitted, auto-assigns the next integer ID."""
    db = load_db()
    if record_id is None:
        existing_ids = []
        for rid in db.keys():
            try:
                existing_ids.append(int(rid))
            except ValueError:
                pass
        record_id = str((max(existing_ids) + 1) if existing_ids else 1)

    record_id = str(record_id)
    if record_id in db:
        raise ValueError(f"Record ID '{record_id}' already exists.")

    record = _normalize_record(record)
    db[record_id] = record
    save_db(db)
    return record_id


def update_user(record_id: str, updates: Dict[str, Any]) -> Dict[str, Any]:
    """Merge `updates` into the existing record (shallow merge for top-level,
    deep merge for nested 'address'/'business'/'income' dicts)."""
    db = load_db()
    record_id = str(record_id)
    if record_id not in db:
        raise KeyError(f"Record ID '{record_id}' not found.")

    record = db[record_id]
    for key, value in updates.items():
        if isinstance(value, dict) and isinstance(record.get(key), dict):
            record[key].update(value)
        else:
            record[key] = value

    record = _normalize_record(record)
    db[record_id] = record
    save_db(db)
    return record


def delete_user(record_id: str) -> None:
    db = load_db()
    record_id = str(record_id)
    if record_id not in db:
        raise KeyError(f"Record ID '{record_id}' not found.")
    del db[record_id]
    save_db(db)


def replace_user(record_id: str, record: Dict[str, Any]) -> Dict[str, Any]:
    """Fully overwrite a record with new data."""
    db = load_db()
    record_id = str(record_id)
    record = _normalize_record(record)
    db[record_id] = record
    save_db(db)
    return record


def _normalize_record(record: Dict[str, Any]) -> Dict[str, Any]:
    """Ensure required nested structures exist with sane defaults."""
    record.setdefault("address", {})
    for k in ("line1", "city", "state", "pin", "country"):
        record["address"].setdefault(k, None)

    record.setdefault("business", {})
    for k in ("cin", "gst", "tan", "name", "type"):
        record["business"].setdefault(k, None)

    record.setdefault("income", {})
    for k in ("annual_declared", "employer", "employment_type"):
        record["income"].setdefault(k, None)

    for k in ("is_pep", "is_sanctioned", "risk_flag"):
        record.setdefault(k, False)

    for k in ("aadhaar", "pan", "passport", "driving_licence", "name", "dob",
              "gender", "nationality", "phone", "email"):
        record.setdefault(k, None)

    return record


# --------------------------------------------------------------------------
# Verification helpers (used by the KYC pipeline)
# --------------------------------------------------------------------------
def find_record_by_id_number(id_type: str, id_number: str, db: Optional[Dict[str, Any]] = None) -> Optional[Tuple[str, Dict[str, Any]]]:
    """
    Find a database record whose ID field of the given type matches id_number.

    id_type: one of 'aadhaar', 'pan', 'passport', 'driving_licence' (or 'dl')
    Returns (record_id, record) or None if not found.
    """
    if not id_number:
        return None

    field = ID_FIELD_MAP.get(id_type.lower())
    if not field:
        return None

    db = db if db is not None else load_db()
    id_number_norm = str(id_number).strip().upper()

    for rid, record in db.items():
        value = record.get(field)
        if value and str(value).strip().upper() == id_number_norm:
            return rid, record

    return None


def find_record_by_name(name: str, db: Optional[Dict[str, Any]] = None) -> List[Tuple[str, Dict[str, Any]]]:
    """Find database records with a fuzzy/normalized name match (case-insensitive substring)."""
    if not name:
        return []
    db = db if db is not None else load_db()
    name_norm = name.strip().lower()
    matches = []
    for rid, record in db.items():
        rec_name = (record.get("name") or "").strip().lower()
        if not rec_name:
            continue
        if rec_name == name_norm or name_norm in rec_name or rec_name in name_norm:
            matches.append((rid, record))
    return matches


def verify_against_registry(extracted_fields: Dict[str, Any], db: Optional[Dict[str, Any]] = None) -> Dict[str, Any]:
    """
    Cross-check a customer's extracted document fields against the mock
    DigiLocker-style registry to determine whether the user / documents
    appear genuine.

    extracted_fields: dict possibly containing keys like
        full_name, date_of_birth, id_number, nationality, address,
        and an optional 'id_type' hint (aadhaar/pan/passport/driving_licence)

    Returns a dict:
        {
          "matched": bool,
          "matched_record_id": str | None,
          "match_method": "id_number" | "name" | "none",
          "field_matches": {field: bool, ...},
          "registry_record": dict | None,
          "is_pep": bool,
          "is_sanctioned": bool,
          "risk_flag": bool,
          "discrepancies": [str, ...],
          "verdict": "genuine" | "mismatch" | "not_found",
        }
    """
    db = db if db is not None else load_db()

    id_number = extracted_fields.get("id_number")
    full_name = extracted_fields.get("full_name")
    dob = extracted_fields.get("date_of_birth")

    record_match = None
    match_method = "none"

    # 1) Try matching by ID number across all known ID types
    if id_number:
        for id_type in ("aadhaar", "pan", "passport", "driving_licence"):
            found = find_record_by_id_number(id_type, id_number, db)
            if found:
                record_match = found
                match_method = "id_number"
                break

    # 2) Fall back to name match
    if record_match is None and full_name:
        name_matches = find_record_by_name(full_name, db)
        if len(name_matches) == 1:
            record_match = name_matches[0]
            match_method = "name"
        elif len(name_matches) > 1:
            # ambiguous - pick first but note it
            record_match = name_matches[0]
            match_method = "name_ambiguous"

    if record_match is None:
        return {
            "matched": False,
            "matched_record_id": None,
            "match_method": "none",
            "field_matches": {},
            "registry_record": None,
            "is_pep": False,
            "is_sanctioned": False,
            "risk_flag": False,
            "discrepancies": ["No matching record found in identity registry."],
            "verdict": "not_found",
        }

    record_id, record = record_match
    discrepancies: List[str] = []
    field_matches: Dict[str, bool] = {}

    # Name comparison (token overlap, tolerant of suffixes/prefixes)
    if full_name:
        reg_name = (record.get("name") or "").strip().lower()
        extr_name = full_name.strip().lower()
        reg_tokens = set(reg_name.split())
        extr_tokens = set(extr_name.split())
        overlap = len(reg_tokens & extr_tokens) / max(len(reg_tokens | extr_tokens), 1)
        name_match = overlap >= 0.5
        field_matches["name"] = name_match
        if not name_match:
            discrepancies.append(f"Name mismatch: document='{full_name}' vs registry='{record.get('name')}'.")

    # DOB comparison
    if dob:
        reg_dob = record.get("dob")
        dob_match = (reg_dob == dob)
        field_matches["date_of_birth"] = dob_match
        if not dob_match:
            discrepancies.append(f"Date of birth mismatch: document='{dob}' vs registry='{reg_dob}'.")

    # Nationality comparison
    nationality = extracted_fields.get("nationality")
    if nationality:
        reg_nat = (record.get("nationality") or "").strip().lower()
        nat_match = reg_nat == nationality.strip().lower()
        field_matches["nationality"] = nat_match
        if not nat_match:
            discrepancies.append(f"Nationality mismatch: document='{nationality}' vs registry='{record.get('nationality')}'.")

    if match_method == "name_ambiguous":
        discrepancies.append("Multiple registry records share a similar name; ID-number match recommended for certainty.")

    verdict = "genuine" if not discrepancies else "mismatch"

    return {
        "matched": True,
        "matched_record_id": record_id,
        "match_method": match_method,
        "field_matches": field_matches,
        "registry_record": record,
        "is_pep": bool(record.get("is_pep", False)),
        "is_sanctioned": bool(record.get("is_sanctioned", False)),
        "risk_flag": bool(record.get("risk_flag", False)),
        "discrepancies": discrepancies,
        "verdict": verdict,
    }


## File: `db/generate_dummy_db.py`

In [ ]:
"""
generate_dummy_db.py
One-off generator script for db/users_db.json — a mock "DigiLocker-style"
government identity database used to verify whether customer-submitted
document data is genuine (matches official records) or fake/mismatched.

Run:  python db/generate_dummy_db.py
"""

import json
import os
import random
import string

random.seed(42)

FIRST_NAMES = [
    "Priya", "Rahul", "Anjali", "Vikram", "Sneha", "Arjun", "Kavya", "Rohan",
    "Divya", "Karthik", "Pooja", "Amit", "Neha", "Suresh", "Meera", "Aditya",
    "Lakshmi", "Sanjay", "Ritu", "Manoj",
]
LAST_NAMES = [
    "Sharma", "Verma", "Iyer", "Reddy", "Nair", "Gupta", "Singh", "Patel",
    "Menon", "Das", "Banerjee", "Rao", "Mehta", "Joshi", "Kapoor", "Pillai",
]
CITIES_STATES = [
    ("Mumbai", "Maharashtra"), ("Delhi", "Delhi"), ("Bengaluru", "Karnataka"),
    ("Chennai", "Tamil Nadu"), ("Kochi", "Kerala"), ("Hyderabad", "Telangana"),
    ("Pune", "Maharashtra"), ("Kolkata", "West Bengal"), ("Jaipur", "Rajasthan"),
    ("Ahmedabad", "Gujarat"), ("Lucknow", "Uttar Pradesh"), ("Chandigarh", "Punjab"),
]
STREET_NAMES = ["Park Street", "MG Road", "Church Street", "Station Road", "Gandhi Nagar", "Lake View Lane", "Hill Road", "Civil Lines"]
EMPLOYERS = ["HCL", "TCS", "Infosys", "Wipro", "Self Employed", "Reliance Industries", "ICICI Bank", "Tata Motors", "Govt of India", "Startup Pvt Ltd"]
EMPLOYMENT_TYPES = ["Salaried", "Business", "Self-Employed", "Unemployed", "Retired"]
GENDERS = ["Male", "Female", "Other"]
BUSINESS_TYPES = ["Pvt Ltd", "LLP", "Proprietorship", "Partnership", None]


def rand_alnum(n):
    return "".join(random.choices(string.ascii_uppercase + string.digits, k=n))


def rand_digits(n):
    return "".join(random.choices(string.digits, k=n))


def make_record(idx: int, force_pep=False, force_sanctioned=False, force_risk=False):
    first = random.choice(FIRST_NAMES)
    last = random.choice(LAST_NAMES)
    name = f"{first} {last}"
    city, state = random.choice(CITIES_STATES)

    dob_year = random.randint(1955, 2005)
    dob_month = random.randint(1, 12)
    dob_day = random.randint(1, 28)
    dob = f"{dob_year:04d}-{dob_month:02d}-{dob_day:02d}"

    has_business = random.random() < 0.4
    business_type = random.choice(BUSINESS_TYPES) if has_business else None

    record = {
        "aadhaar": f"AADHAAR-{rand_alnum(10)}",
        "pan": f"PAN-{rand_alnum(10)}",
        "passport": f"PASS-{rand_alnum(8)}",
        "driving_licence": f"DL-{rand_alnum(8)}",
        "name": name,
        "dob": dob,
        "gender": random.choice(GENDERS),
        "address": {
            "line1": f"{random.randint(1, 999)} {random.choice(STREET_NAMES)}",
            "city": city,
            "state": state,
            "pin": rand_digits(6),
            "country": "India",
        },
        "nationality": "Indian",
        "phone": f"+91{rand_digits(10)}",
        "email": f"{first.lower()}_{last.lower()}{idx}@email.com",
        "business": {
            "cin": f"CIN-{rand_alnum(15)}" if business_type == "Pvt Ltd" else None,
            "gst": f"GST-{rand_alnum(13)}" if has_business else None,
            "tan": f"TAN-{rand_alnum(8)}" if has_business else None,
            "name": f"{last} {random.choice(['Enterprises', 'Traders', 'Solutions', 'Industries', 'Exports'])}" if has_business else None,
            "type": business_type,
        },
        "income": {
            "annual_declared": random.choice([300000, 500000, 800000, 1200000, 2500000, 5000000, 8000000, 15000000]),
            "employer": random.choice(EMPLOYERS),
            "employment_type": random.choice(EMPLOYMENT_TYPES),
        },
        "is_pep": force_pep,
        "is_sanctioned": force_sanctioned,
        "risk_flag": force_risk,
    }
    return record


def main():
    db = {}

    # 1) Standard "clean" records
    for i in range(1, 31):
        db[str(i)] = make_record(i)

    # 2) A few flagged records for testing screening logic
    db["31"] = make_record(31, force_pep=True)
    db["31"]["name"] = "Jane Smith"  # matches MOCK_WATCHLIST PEP entry

    db["32"] = make_record(32, force_sanctioned=True)
    db["32"]["name"] = "John Doe"  # matches MOCK_WATCHLIST sanctions entry

    db["33"] = make_record(33, force_risk=True)
    db["33"]["name"] = "Carlos Mendez"  # matches MOCK_WATCHLIST adverse media entry

    db["34"] = make_record(34, force_sanctioned=True)
    db["34"]["name"] = "Vladimir Petrov"
    db["34"]["nationality"] = "Russian"
    db["34"]["address"]["country"] = "Russia"

    # 3) A minor (for age plausibility testing)
    db["35"] = make_record(35)
    db["35"]["name"] = "Aditya Kapoor"
    db["35"]["dob"] = "2015-04-12"

    # 4) An expired-looking / incomplete record
    db["36"] = make_record(36)
    db["36"]["name"] = "Suresh Singh"
    db["36"]["passport"] = None
    db["36"]["driving_licence"] = None

    # Sample provided in the prompt (kept for compatibility)
    db["1"] = {
        "aadhaar": "AADHAAR-UP75NWH376Q1",
        "pan": "PAN-GSDQW8FIIT",
        "passport": "PASS-TE1DVFMH",
        "driving_licence": "DL-JHFN07RVS3",
        "name": "Priya Sharma 1",
        "dob": "1965-10-30",
        "gender": "Other",
        "address": {
            "line1": "686 Park Street",
            "city": "Kochi",
            "state": "Telangana",
            "pin": "245985",
            "country": "India",
        },
        "nationality": "Indian",
        "phone": "+919336942245",
        "email": "priya_sharma1@email.com",
        "business": {
            "cin": None,
            "gst": "GST-JLLQXBP6FBD5F1N",
            "tan": "TAN-A4TG0JWF",
            "name": None,
            "type": "LLP",
        },
        "income": {
            "annual_declared": 5000000,
            "employer": "HCL",
            "employment_type": "Business",
        },
        "is_pep": False,
        "is_sanctioned": False,
        "risk_flag": False,
    }

    out_path = os.path.join(os.path.dirname(__file__), "users_db.json")
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(db, f, indent=2)

    print(f"Generated {len(db)} records -> {out_path}")


if __name__ == "__main__":
    main()


## File: `main.py`

In [ ]:
"""
Agentic KYC Intelligence Platform - Streamlit Frontend.

Multi-agent, AMD-orchestrated end-to-end customer due diligence:
 - Document upload (identity, address, live photo, business docs)
 - Portrait photo is AUTO-EXTRACTED from the identity document —
   no separate "Photo on ID" upload required from the user.
 - Pipeline execution with live agent-by-agent progress
 - Explainable KYC decision (APPROVE / REVIEW / ESCALATE) with full evidence
 - Human-in-the-loop override & audit trail viewer

Change log:
 - Removed `photo_on_id_proof` / "Photo on ID" file uploader entirely.
 - `build_documents_dict()` now skips any category whose key contains
   "photo_id" or "photo_on_id" so the change is config-driven.
 - Added an info banner on the New Onboarding page explaining the
   auto-extraction behaviour to the operator/reviewer.
 - Required-document validation guard updated to exclude the removed category.
 - About page copy updated to reflect automatic portrait extraction.
"""

import json
import os
import uuid
from datetime import datetime

import pandas as pd
import streamlit as st

from config import DOCUMENT_CATEGORIES, AUDIT_LOG_FILE, RISK_THRESHOLDS
from agents import KYCOrchestrator

# --------------------------------------------------------------------------
# Page config & styling
# --------------------------------------------------------------------------
st.set_page_config(
    page_title="Agentic KYC Intelligence Platform",
    page_icon="🛡️",
    layout="wide",
    initial_sidebar_state="expanded",
)

DECISION_COLORS = {
    "APPROVE": "#1f9d55",
    "REVIEW": "#d97706",
    "ESCALATE": "#dc2626",
}

STATUS_ICONS = {
    "success": "✅",
    "warning": "⚠️",
    "error": "❌",
}

CUSTOM_CSS = """
<style>
.decision-badge {
    display: inline-block;
    padding: 0.5rem 1.25rem;
    border-radius: 0.5rem;
    font-weight: 700;
    font-size: 1.4rem;
    color: white;
    text-align: center;
}
.agent-card {
    border: 1px solid #e5e7eb;
    border-radius: 0.5rem;
    padding: 1rem;
    margin-bottom: 0.75rem;
    background-color: #fafafa;
}
.evidence-item {
    font-size: 0.9rem;
    margin: 0.15rem 0;
}
.metric-box {
    border: 1px solid #e5e7eb;
    border-radius: 0.5rem;
    padding: 0.75rem;
    text-align: center;
}
</style>
"""
st.markdown(CUSTOM_CSS, unsafe_allow_html=True)

# --------------------------------------------------------------------------
# Keys that must NEVER appear as file uploaders — portrait is auto-extracted
# --------------------------------------------------------------------------
_AUTO_EXTRACTED_CATEGORIES = {"photo_on_id_proof", "photo_id", "photo_on_id"}


def _is_auto_extracted(category_key: str) -> bool:
    """Return True for categories whose content is derived, not uploaded."""
    return category_key.lower() in _AUTO_EXTRACTED_CATEGORIES


def _is_live_capture(category_key: str, label: str) -> bool:
    """Return True for categories that require a live camera capture."""
    live_keys = {"live_photo", "selfie", "live_selfie", "liveness"}
    return (
        category_key.lower() in live_keys
        or "live" in label.lower()
        or "selfie" in label.lower()
        or "photo verification" in label.lower()
    )


# --------------------------------------------------------------------------
# Session state init
# --------------------------------------------------------------------------
if "orchestrator" not in st.session_state:
    st.session_state.orchestrator = KYCOrchestrator()
if "report" not in st.session_state:
    st.session_state.report = None
if "customer_id" not in st.session_state:
    st.session_state.customer_id = f"CUST-{uuid.uuid4().hex[:8].upper()}"
if "pipeline_log" not in st.session_state:
    st.session_state.pipeline_log = []

# --------------------------------------------------------------------------
# Sidebar - navigation & customer info
# --------------------------------------------------------------------------
with st.sidebar:
    st.title("🛡️ Agentic KYC Platform")
    st.caption("AMD-orchestrated multi-agent due diligence")

    page = st.radio(
        "Navigate",
        ["📤 New Onboarding", "📊 Decision Report", "🗂️ Audit Trail", "ℹ️ About"],
        index=0,
    )

    st.divider()
    st.text_input("Customer ID", key="customer_id")

    st.divider()
    st.caption("Decision Thresholds")
    st.write(f"APPROVE: score ≤ {RISK_THRESHOLDS['approve_max']}")
    st.write(f"REVIEW: ≤ {RISK_THRESHOLDS['review_max']}")
    st.write(f"ESCALATE: > {RISK_THRESHOLDS['review_max']}")

# --------------------------------------------------------------------------
# Helper: file uploader → bytes dict
# --------------------------------------------------------------------------
def build_documents_dict() -> dict:
    """
    Render upload / camera widgets for each document category and return a
    dict keyed by category name.

    Categories in `_AUTO_EXTRACTED_CATEGORIES` are silently skipped — their
    content is derived automatically inside the Identity Verification Agent
    (portrait extracted from the identity_proof document).
    """
    documents = {}

    for category, meta in DOCUMENT_CATEGORIES.items():

        # ── Skip auto-extracted categories — no uploader shown ────────────
        if _is_auto_extracted(category):
            continue

        required_label = " *(required)*" if meta["required"] else " *(optional)*"
        st.markdown(f"**{meta['label']}**{required_label}")

        col1, col2 = st.columns([2, 1])

        with col2:
            subtype = st.selectbox(
                "Type",
                meta["accepted"],
                key=f"subtype_{category}",
                label_visibility="collapsed",
            )

        with col1:
            if _is_live_capture(category, meta["label"]):
                # Live camera capture — uploads disabled for liveness assurance
                st.info("📷 Live capture required — file uploads disabled for security.")
                captured = st.camera_input(
                    f"Capture {meta['label']}",
                    key=f"camera_{category}",
                    help="Click 'Take Photo' to enable your camera.",
                )
                if captured is not None:
                    file_bytes = captured.getvalue()
                    documents[category] = {
                        "filename": (
                            f"{category}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.jpg"
                        ),
                        "bytes": file_bytes,
                        "doc_subtype": subtype,
                        "mimetype": "image/jpeg",
                    }
                    st.success("✓ Live photo captured.")

            else:
                # Standard file upload for identity / address / business docs
                uploaded_file = st.file_uploader(
                    f"Upload {meta['label']}",
                    type=["pdf", "png", "jpg", "jpeg", "bmp", "tiff", "webp"],
                    key=f"upload_{category}",
                    label_visibility="collapsed",
                )
                if uploaded_file is not None:
                    file_bytes = uploaded_file.getvalue()
                    documents[category] = {
                        "filename": uploaded_file.name,
                        "bytes": file_bytes,
                        "doc_subtype": subtype,
                        "mimetype": uploaded_file.type or "",
                    }
                    if uploaded_file.type and "image" in uploaded_file.type:
                        st.image(file_bytes, width=180)
                    else:
                        st.caption(
                            f"📄 {uploaded_file.name} "
                            f"({len(file_bytes) / 1024:.1f} KB)"
                        )

        st.markdown("---")

    return documents


# --------------------------------------------------------------------------
# PAGE: New Onboarding
# --------------------------------------------------------------------------
if page == "📤 New Onboarding":
    st.header("New Customer Onboarding")
    st.write(
        "Upload the customer's documents below. The AMD orchestrator will route "
        "them through the specialised KYC agent pipeline: data extraction, "
        "enrichment, identity verification, compliance screening, and financial profiling."
    )

    # ── Auto-extraction notice ─────────────────────────────────────────────
    st.info(
        "🤖 **Portrait auto-extraction enabled** — the Identity Verification Agent "
        "will automatically detect and extract the portrait photo embedded in the "
        "Identity Proof document. No separate 'Photo on ID' upload is required.",
        icon="ℹ️",
    )

    documents = build_documents_dict()

    submitted = st.button(
        "🚀 Run KYC Pipeline", use_container_width=True, type="primary"
    )

    if submitted:
        # Validate required documents (photo_on_id excluded — it's auto-derived)
        missing_required = [
            meta["label"]
            for cat, meta in DOCUMENT_CATEGORIES.items()
            if meta["required"]
            and not _is_auto_extracted(cat)
            and cat not in documents
        ]
        if missing_required:
            st.error(f"Missing required documents: {', '.join(missing_required)}")
        else:
            st.session_state.pipeline_log = []
            progress_bar = st.progress(0, text="Starting AMD orchestration…")
            steps_total = 6
            step_counter = {"n": 0}

            log_container = st.container()

            def progress_callback(step_name, agent_result):
                step_counter["n"] += 1
                pct = int((step_counter["n"] / steps_total) * 100)
                progress_bar.progress(pct, text=f"Completed: {step_name}")
                icon = STATUS_ICONS.get(agent_result.status, "ℹ️")
                st.session_state.pipeline_log.append(
                    (step_name, agent_result.status, agent_result.risk_score)
                )
                with log_container:
                    st.write(
                        f"{icon} **{step_name}** — "
                        f"risk score: {agent_result.risk_score:.1f} — "
                        f"{agent_result.explanation}"
                    )

            with st.spinner("Running multi-agent KYC pipeline…"):
                report = st.session_state.orchestrator.run_pipeline(
                    customer_id=st.session_state.customer_id,
                    documents=documents,
                    progress_callback=progress_callback,
                )
            st.session_state.report = report
            progress_bar.progress(100, text="Pipeline complete.")
            st.success(
                "KYC pipeline completed. "
                "Go to '📊 Decision Report' to review the explainable decision."
            )

# --------------------------------------------------------------------------
# PAGE: Decision Report
# --------------------------------------------------------------------------
elif page == "📊 Decision Report":
    st.header("Explainable KYC Decision Report")

    report = st.session_state.report
    if not report:
        st.info("No report yet. Run a new onboarding from '📤 New Onboarding'.")
    else:
        decision = report.get("final_decision", report["decision"])
        color = DECISION_COLORS.get(decision, "#6b7280")

        col1, col2, col3 = st.columns([1.2, 1, 1])
        with col1:
            st.markdown(
                f"<div class='decision-badge' style='background-color:{color};'>"
                f"{decision}</div>",
                unsafe_allow_html=True,
            )
        with col2:
            st.markdown(
                f"<div class='metric-box'><b>Composite Risk Score</b><br>"
                f"<span style='font-size:1.5rem;'>"
                f"{report['composite_risk_score']:.1f} / 100</span></div>",
                unsafe_allow_html=True,
            )
        with col3:
            st.markdown(
                f"<div class='metric-box'><b>Customer ID</b><br>"
                f"<span style='font-size:1.1rem;'>{report['customer_id']}</span></div>",
                unsafe_allow_html=True,
            )

        st.markdown("### 🧭 AMD Final Explanation")
        st.write(report["final_explanation"])

        st.markdown("### 🤖 Agent-by-Agent Findings")
        for label, agent_data in report["agent_results"].items():
            icon = STATUS_ICONS.get(agent_data["status"], "ℹ️")
            with st.expander(
                f"{icon} {agent_data['agent_name']}  —  "
                f"risk: {agent_data['risk_score']:.1f}/100"
            ):
                st.write(f"**Explanation:** {agent_data['explanation']}")

                # Surface face extraction status badge if present
                face_ext = agent_data.get("findings", {}).get("face_extraction_status")
                if face_ext:
                    badge_color = "#1f9d55" if face_ext == "success" else "#dc2626"
                    st.markdown(
                        f"<span style='background:{badge_color};color:white;"
                        f"padding:2px 8px;border-radius:4px;font-size:0.8rem;'>"
                        f"Portrait extraction: {face_ext}</span>",
                        unsafe_allow_html=True,
                    )

                st.write("**Evidence:**")
                for ev in agent_data["evidence"]:
                    st.markdown(
                        f"<div class='evidence-item'>• {ev}</div>",
                        unsafe_allow_html=True,
                    )
                with st.popover("View raw findings (JSON)"):
                    st.json(agent_data["findings"])

        st.markdown("### 📈 Risk Score Breakdown")
        chart_data = pd.DataFrame({
            "Agent": [v["agent_name"] for v in report["agent_results"].values()],
            "Risk Score": [v["risk_score"] for v in report["agent_results"].values()],
        })
        st.bar_chart(chart_data.set_index("Agent"))

        st.markdown("### 👤 Human-in-the-Loop Override")
        if report.get("human_override"):
            ho = report["human_override"]
            st.success(
                f"Reviewed by {ho['reviewer']} at {ho['timestamp']} "
                f"→ Final decision: **{ho['decision']}**"
            )
            st.caption(f"Comment: {ho['comment']}")
        else:
            with st.form("override_form"):
                reviewer = st.text_input("Reviewer name")
                override_decision = st.selectbox(
                    "Override decision",
                    ["APPROVE", "REVIEW", "ESCALATE"],
                    index=["APPROVE", "REVIEW", "ESCALATE"].index(decision),
                )
                comment = st.text_area("Reviewer comment / rationale")
                override_submitted = st.form_submit_button("Submit Review Decision")

            if override_submitted:
                if not reviewer:
                    st.error("Please enter a reviewer name.")
                else:
                    updated = st.session_state.orchestrator.apply_human_override(
                        report, reviewer, override_decision, comment
                    )
                    st.session_state.report = updated
                    st.rerun()

        st.markdown("### 📥 Export")
        report_json = json.dumps(report, indent=2, default=str)
        st.download_button(
            "Download Full Report (JSON)",
            data=report_json,
            file_name=f"kyc_report_{report['customer_id']}.json",
            mime="application/json",
        )

# --------------------------------------------------------------------------
# PAGE: Audit Trail
# --------------------------------------------------------------------------
elif page == "🗂️ Audit Trail":
    st.header("Audit Trail")
    st.write(
        "Append-only log of every agent action, decision, and human override — "
        "for explainability and compliance."
    )

    if not os.path.exists(AUDIT_LOG_FILE):
        st.info("No audit log entries yet.")
    else:
        records = []
        with open(AUDIT_LOG_FILE, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    records.append(json.loads(line))
                except json.JSONDecodeError:
                    continue

        if not records:
            st.info("No audit log entries yet.")
        else:
            df = pd.DataFrame([
                {
                    "Timestamp": r.get("timestamp"),
                    "Agent": r.get("agent"),
                    "Customer ID": r.get("customer_id"),
                    "Status": r.get("payload", {}).get("status", ""),
                    "Risk Score": r.get("payload", {}).get("risk_score", ""),
                    "Decision": r.get("payload", {}).get(
                        "decision",
                        r.get("payload", {}).get("composite_risk_score", ""),
                    ),
                }
                for r in records
            ])
            st.dataframe(df, use_container_width=True, height=400)

            with st.expander("View raw log entries"):
                for r in reversed(records[-50:]):
                    st.json(r)

# --------------------------------------------------------------------------
# PAGE: About
# --------------------------------------------------------------------------
elif page == "ℹ️ About":
    st.header("About this Platform")
    st.markdown("""
**Agentic KYC Intelligence Platform** demonstrates a multi-agent architecture
for end-to-end customer due diligence, orchestrated by an AMD (Agent Master / Director).

**Pipeline:**
1. **Data & Document Extraction Agent** — OCR + LLM extraction of structured fields from
   identity proof, address proof, and business documents; cross-document consistency checks.
2. **Enrichment Agent** — Normalizes profile data, classifies country/jurisdiction risk,
   checks age plausibility, and computes profile completeness.
3. **Identity Verification Agent** — **Automatically extracts** the portrait photo embedded
   in the identity document (Passport, Aadhaar, Driving Licence, etc.) using OpenCV face
   detection, then compares it against the live captured photo. No separate "Photo on ID"
   upload is required. Also validates document expiry and runs an LLM-based authenticity review.
4. **Compliance Screening Agent** — Screens the customer name against sanctions / PEP /
   adverse media watchlists with fuzzy matching and LLM-assisted true/false-positive analysis.
5. **Financial Profile Agent** — Assesses source-of-funds plausibility and flags unusual
   financial patterns from bank statements / business documents.

**Decision Layer (AMD):** Aggregates weighted risk scores from all agents into a composite
score (0–100), applies configurable thresholds to produce **APPROVE / REVIEW / ESCALATE**,
and generates a plain-language explanation for human reviewers.

**Human Oversight:** Every decision can be reviewed and overridden by a compliance officer,
with the override recorded in the immutable audit trail.

**Explainability:** Every agent records evidence statements and structured findings,
all persisted to an append-only JSONL audit log (`logs/audit_trail.jsonl`).
""")
    st.caption(
        "Built for hackathon demo purposes. Replace mock watchlists, country-risk tables, "
        "and face-matching with production-grade services before real-world use."
    )

## File: `readme.txt`

```text
# Agentic KYC Intelligence Platform

A multi-agent, AMD-orchestrated platform for end-to-end customer due diligence:
data extraction, enrichment, identity verification, compliance screening,
financial profiling, and explainable decisioning with human oversight.

## Architecture

```
Customer Onboarding Request
        │
        ▼
 ┌─────────────────────────┐
 │   AMD Orchestrator       │   (agents/orchestrator.py)
 └─────────────────────────┘
        │
   ┌────┼────────────────────────────────────────────────┐
   ▼    ▼               ▼                ▼                ▼
 Data   Enrichment   Identity        Compliance      Financial
 Extract Agent       Verification    Screening       Profile
 Agent               Agent           Agent           Agent
   │       │              │               │                │
   └───────┴──────────────┴───────────────┴────────────────┘
                          │
                          ▼
        Composite Risk Score → APPROVE / REVIEW / ESCALATE
                          │
                          ▼
            Explainable Report + Evidence Trail
                          │
                          ▼
              Human-in-the-loop Override (Streamlit UI)
                          │
                          ▼
              Append-only Audit Log (logs/audit_trail.jsonl)
```

## Project Structure

```
kyc_platform/
├── main.py                      # Streamlit application (entry point)
├── config.py                    # LLM/embedding config, thresholds, watchlist, doc categories
├── req.txt                      # Python dependencies
├── readme.txt                   # This file
├── admin.py                      # Streamlit admin panel for the mock identity registry DB
├── agents/
│   ├── __init__.py
│   ├── base_agent.py            # Shared AgentResult + BaseAgent (LLM calls, audit logging)
│   ├── doc_utils.py              # OCR, PDF parsing, face comparison, image quality checks
│   ├── data_extraction_agent.py  # Agent 1: document parsing & cross-checks
│   ├── enrichment_agent.py       # Agent 2: profile normalization & country risk
│   ├── identity_verification_agent.py  # Agent 3: face match, expiry, authenticity
│   ├── registry_verification_agent.py  # Agent 4: cross-check vs mock DigiLocker registry
│   ├── screening_agent.py        # Agent 5: sanctions/PEP/adverse media screening
│   ├── financial_profile_agent.py# Agent 6: source-of-funds & financial behavior
│   └── orchestrator.py           # AMD: pipeline coordination + composite scoring + decision
├── db/
│   ├── __init__.py
│   ├── db_manager.py             # CRUD + registry verification helper functions
│   ├── generate_dummy_db.py       # One-off script to (re)generate users_db.json
│   └── users_db.json              # Mock "DigiLocker-style" identity registry (36 sample records)
├── uploads/                       # (runtime) uploaded documents, if persisted
└── logs/
    └── audit_trail.jsonl          # (runtime) append-only audit log
```

## Setup

1. **Python environment** (Python 3.10+ recommended):

```bash
python -m venv venv
source venv/bin/activate   # Windows: venv\Scripts\activate
pip install -r req.txt
```

2. **System dependencies** (for OCR and PDF/image processing):

- **Tesseract OCR** (required for `pytesseract`):
  - Ubuntu/Debian: `sudo apt-get install tesseract-ocr`
  - macOS: `brew install tesseract`
  - Windows: install from https://github.com/UB-Mannheim/tesseract/wiki and add to PATH

- **OpenCV** is installed via `opencv-python-headless` (no extra system deps needed
  on most platforms).

3. **Configure API credentials**

Edit `config.py` or set environment variables:

```bash
export GENAI_BASE_URL="https://genailab.tcs.in"
export GENAI_API_KEY="YOUR_REAL_API_KEY"
export LLM_MODEL_NAME="azure_ai/genailab-maas-DeepSeek-V3-0324"
export EMBEDDING_MODEL_NAME="azure/genailab-maas-text-embedding-3-large"
```

> ⚠️ Never commit real API keys to source control. Use environment variables
> or a secrets manager in production.

## Running the App

```bash
streamlit run main.py
```

Open the URL shown (typically http://localhost:8501).

## Using the Platform

1. **New Onboarding tab**:
   - Upload required documents:
     - **Identity Proof**: Passport / Government ID / Driving License
     - **Address Proof**: Government Utility Bill / Aadhaar / Bank Statement
     - **Photo on ID Proof**: cropped/extracted photo from the ID document
     - **Live Captured Photo**: live selfie/webcam capture
     - **Business Document** (optional): Tax Invoice / Business Registration / Lease Agreement
   - Click **"Run KYC Pipeline"**. Each agent runs sequentially with live progress
     and intermediate risk scores shown.

2. **Decision Report tab**:
   - View the composite risk score, final decision (APPROVE / REVIEW / ESCALATE),
     and an AMD-generated plain-language explanation.
   - Expand each agent's card to see detailed evidence and raw findings (JSON).
   - View a bar chart comparing each agent's risk contribution.
   - **Human-in-the-loop**: a compliance reviewer can submit a final override
     decision with a comment, recorded permanently in the audit trail.
   - Export the full report as JSON.

3. **Audit Trail tab**:
   - View the complete append-only log of every agent action and decision
     across all customers/sessions, sourced from `logs/audit_trail.jsonl`.

## Configuration & Tuning (config.py)

- `DOCUMENT_CATEGORIES` — document types accepted, labels, and required/optional flags.
- `RISK_THRESHOLDS` — composite score cutoffs for APPROVE / REVIEW / ESCALATE.
- `AGENT_WEIGHTS` — weight of each agent's risk score in the composite calculation.
- `MOCK_WATCHLIST` — sample sanctions/PEP/adverse-media list (replace with real
  data feeds, e.g. OFAC, UN, EU consolidated lists, Dow Jones, Refinitiv, etc.).
- `FACE_MATCH_THRESHOLD` — minimum face similarity (0-1) to pass identity verification.
- `HIGH_RISK_COUNTRIES` / `MEDIUM_RISK_COUNTRIES` (in `enrichment_agent.py`) —
  jurisdiction risk classification; replace with FATF grey/black lists in production.

## Mock Identity Registry ("DigiLocker") & Admin Panel

The platform includes a **mock government identity registry** (`db/users_db.json`)
containing 36 sample records (Aadhaar, PAN, Passport, Driving Licence, name, DOB,
address, business details, income/employment, and PEP/sanctions/risk flags).

### Registry Verification Agent

A dedicated **Registry Verification Agent** (`agents/registry_verification_agent.py`)
cross-checks the customer's extracted identity-document fields (ID number, name,
DOB, nationality) against this registry:

- **`genuine`** — matched record found and all compared fields agree.
- **`mismatch`** — a record was matched but one or more fields (name/DOB/nationality)
  differ from the official record.
- **`not_found`** — no matching record exists for the submitted ID number/name,
  signaling a potentially **fake or unregistered identity** (high risk, forces escalation).

It also surfaces any pre-existing `is_pep`, `is_sanctioned`, or `risk_flag` markers
on the matched registry record.

### Admin Panel (`admin.py`)

Run a separate Streamlit app to manage the registry:

```bash
streamlit run admin.py --server.port 8502
```

Features:
- **Browse Records** — searchable table of all registry entries with PEP/Sanctions/Risk indicators.
- **Add Record** — create a new identity record (auto-assigns the next integer ID, or specify a custom ID).
- **Edit / Delete Record** — full edit form for any record, plus quick-toggle buttons for PEP/Sanctioned/Risk flags, and delete.
- **Export / Raw DB** — download `users_db.json` or inspect it as raw JSON.

### Regenerating the dummy database

```bash
python db/generate_dummy_db.py
```

This regenerates `db/users_db.json` with 36 records: 30 random "clean" customers,
plus several pre-flagged records for testing (a PEP match, a sanctioned match,
an adverse-media match, a minor for age-plausibility checks, and an incomplete record).

## Production Hardening Checklist

- [ ] Replace the heuristic histogram-based face comparison (`doc_utils.compare_faces`)
      with a dedicated face-recognition model/service (e.g. AWS Rekognition, Azure Face API,
      or an open-source face-embedding model like ArcFace/FaceNet).
- [ ] Replace `MOCK_WATCHLIST` with live sanctions/PEP/adverse-media data providers.
- [ ] Add persistent storage (database) for customer records, documents, and reports
      instead of in-memory session state.
- [ ] Add authentication/authorization for reviewer access to the Streamlit app.
- [ ] Encrypt documents at rest and in transit; apply data-retention policies (PII handling).
- [ ] Add rate limiting / retry / circuit breakers around LLM and external API calls.
- [ ] Add structured logging + monitoring/alerting (beyond the JSONL audit log).
- [ ] Add unit/integration tests for each agent and the orchestrator.
- [ ] Validate uploaded file types/sizes and scan for malware before processing.
- [ ] Internationalize OCR (multi-language Tesseract language packs) for global ID documents.

## Extensibility

- **Add a new agent**: create `agents/<name>_agent.py` subclassing `BaseAgent`,
  implement `run()` returning an `AgentResult`, register it in `agents/__init__.py`
  and wire it into `KYCOrchestrator.run_pipeline`.
- **Adjust decisioning**: tune `AGENT_WEIGHTS` and `RISK_THRESHOLDS` in `config.py`
  without touching agent logic.
- **Swap LLM/embedding provider**: update `get_llm` / `get_embedding_model` in `config.py`.

```

## File: `req.txt`

```text
streamlit>=1.33.0
langchain>=0.2.0
langchain-core>=0.2.0
langchain-openai>=0.1.7
httpx>=0.27.0
pandas>=2.0.0
pillow>=10.0.0
pytesseract>=0.3.10
PyMuPDF>=1.23.0
opencv-python-headless>=4.8.0
numpy>=1.24.0

```